In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install networkx momepy tqdm

In [ ]:
!pip install fuzzywuzzy[speedup]

In [ ]:
!pip install "folium>=0.12" matplotlib mapclassify

In [ ]:
!pip install kneed

# **Data Science Libraries Used**

In [ ]:
import os
import time
import geopandas as gpd
import matplotlib.pyplot as plt
from fuzzywuzzy import fuzz
import pandas as pd
import networkx as nx
from shapely.geometry import Point, LineString, MultiLineString,MultiPoint
from shapely.ops import nearest_points, linemerge
from sklearn.cluster import DBSCAN,KMeans
import numpy as np
from scipy.spatial import cKDTree
import warnings
import traceback
from collections import defaultdict
from shapely.ops import split as shapely_split, nearest_points
from tqdm.auto import tqdm
import folium
from folium.plugins import MarkerCluster, Fullscreen, HeatMap
from tqdm.notebook import tqdm
import gc
import math
from kneed import KneeLocator
from sklearn.metrics import silhouette_score
import json
import shutil
import threading
import http.server
import socketserver
import subprocess
import re
import requests
import gradio as gr
from branca.element import MacroElement, Template
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


### Core Libraries
#### geopandas:
Extends the popular pandas library to work with geospatial data. It is the primary tool in this script for reading shapefiles, managing geographic shapes (points, lines), performing spatial operations, and reprojecting data between coordinate systems.
#### pandas:
The foundational data manipulation library in Python. It provides the DataFrame structure that GeoPandas is built upon. We use it for general data handling, filtering (gdf[gdf['service_level'] == 'Underserved']), and data aggregation.
#### matplotlib.pyplot:
A fundamental plotting library in Python. While not used in the provided functions, it is typically imported in analysis scripts like this to create visualizations, such as plotting the clustered points on a map to visually inspect the results.
### Geospatial & Network Analysis Libraries
#### networkx:
 A powerful library for the creation, manipulation, and study of complex networks (graphs). Its central role in this script is to convert the road GeoDataFrame into a mathematical graph, enabling the use of highly optimized algorithms like Dijkstra's to calculate the shortest travel distances between facilities and habitations.
#### shapely:
The underlying engine for geometric operations in GeoPandas. We import specific components:
##### geometry (Point, LineString, MultiLineString, MultiPoint):
These are the classes used to represent the actual geometric objects. For example, Point is used to find coordinates on a line, and MultiPoint is used to split a line at multiple locations.
#### ops (nearest_points, linemerge, shapely_split):
These are functions that perform complex operations on geometries. nearest_points is used to snap habitations to the closest road, and shapely_split is critical for the "graph surgery" step of breaking a road segment into new, smaller segments.
### Machine Learning & Scientific Computing
#### sklearn.cluster.DBSCAN:
From the Scikit-learn library, this is the specific machine learning algorithm used to perform the density-based clustering. It identifies groups of underserved habitations based on the eps (distance) and min_samples (density) parameters.
#### numpy:
 The fundamental package for scientific computing in Python. Its key role here is to efficiently convert the (x, y) coordinates from the GeoDataFrame into a numerical array, which is the required input format for the DBSCAN algorithm.
scipy.spatial.cKDTree: A data structure from the SciPy library that enables extremely fast nearest-neighbor lookups. While not explicitly used in the provided functions, it is often imported for optimizing custom spatial join operations or finding the nearest points between large datasets.
### Utility & Helper Libraries
#### os:
 The standard library for interacting with the operating system. It is used here to create output directories (os.makedirs) and construct file paths in a way that works across different operating systems (os.path.join).
#### time:
Provides simple functions for measuring time. It is used to calculate and report the total execution time of the entire analysis pipeline.
#### fuzzywuzzy:
A library for string matching. In geospatial analysis, it can be useful for data cleaning, such as matching facility names from different sources that may have slight variations in spelling (e.g., "St. Mary's Hospital" vs. "Saint Mary Hospital").
tqdm: A fast, extensible progress bar library. It is wrapped around long-running loops (like splitting thousands of road segments) to provide a visual indicator of progress, making the script's execution much more user-friendly.
warnings: Used to control how Python handles warning messages. This can be used to suppress non-critical warnings (like DeprecationWarning) to keep the output log clean.
#### traceback:
 A standard library module used for error diagnostics. It is called in the except block to print a detailed "stack trace" when the script encounters a critical error, which is invaluable for debugging.
collections.defaultdict: A specialized dictionary that provides a default value for a key that has not been set yet. This can simplify code for grouping items, for instance, by creating an empty list for a cluster ID the first time it is encountered....in this format


# **Pre-Processing Steps**

In [ ]:
shapefile_path_1 = '/content/drive/MyDrive/Rukiya project/Facilities/Facilities.shp'
shapefile_path_2 = '/content/drive/MyDrive/Rukiya project/Habitation/Habitation.shp'
shapefile_path_3 = '/content/drive/MyDrive/Rukiya project/Roads/gis_osm_roads_free_1.shp'


gdf_facilities, gdf_habitation,gdf_roads = None, None ,None
try:
    gdf_facilities = gpd.read_file(shapefile_path_1)
    print(f"Successfully loaded: {shapefile_path_1}")
except Exception as e:
    print(f"Error loading '{shapefile_path_1}': {e}")

try:
    gdf_habitation = gpd.read_file(shapefile_path_2)
    print(f"Successfully loaded: {shapefile_path_2}")
except Exception as e:
    print(f"Error loading '{shapefile_path_2}': {e}")

try:
    gdf_roads = gpd.read_file(shapefile_path_3)
    print(f"Successfully loaded: {shapefile_path_3}")
except Exception as e:
    print(f"Error loading '{shapefile_path_3}': {e}")

In [ ]:
gdf_habitation_with_path = gpd.read_file('/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis_No_Jump_Lines/habitations_with_path_info.shp')

In [ ]:
gdf_habitation_with_path.info()

In [ ]:
print("Facilities",gdf_facilities.head())
print("Habitation",gdf_habitation.head())
print("Roads",gdf_roads.head())

In [ ]:
print("habitation",gdf_facilities.info())
print("facilities",gdf_habitation.info())
print("road",gdf_roads.info())

## Dataset Column Descriptions

### Facilities (Infrastructure Points)
| Column | Description |
|--------|-------------|
| STATE_ID | State identifier (integer) |
| DISTRICT_I | District identifier (integer) |
| BLOCK_ID | Administrative block identifier (integer) |
| HAB_ID | Associated habitation identifier (integer) |
| FACILITY_I | Unique facility ID (integer) |
| FAC_DESC | Facility description (e.g., "NDCC Bank", "Mandi") |
| FAC_CATEGO | Facility category (e.g., "Transport/Admin", "Agro") |
| geometry | Point geometry (latitude/longitude coordinates) |

### Habitation (Settlement Areas)
| Column | Description |
|--------|-------------|
| HAB_ID | Unique habitation ID (integer) |
| STATE_ID | State identifier (integer) |
| DISTRICT_I | District identifier (integer) |
| BLOCK_ID | Block identifier (integer) |
| HAB_NAME | Habitation name (e.g., "Shidhachi shelwadi") |
| TOT_POPULA | Total population (numeric) |
| geometry | Point geometry (latitude/longitude coordinates) |

### Roads (Linear Features)
| Column | Description |
|--------|-------------|
| osm_id | OpenStreetMap feature ID (integer) |
| code | Internal road code (integer) |
| fclass | Road class (e.g., "tertiary", "footway", "residential") |
| name | Road name (string, often null) |
| ref | Road reference (e.g., route number, often null) |
| oneway | One-way indicator ('B' = bi-directional) |
| maxspeed | Maximum speed limit ( 0) |
| layer | Mapping layer (integer, usually 0) |
| bridge | Bridge indicator ('F' = false) |
| tunnel | Tunnel indicator ('F' = false) |
| geometry | LineString geometry (start/end coordinates for road path) |

**Note:** All geometries use EPSG:4326 (WGS84) coordinate system. The datasets appear to contain infrastructure, settlement, and transportation information for areas in India (Maharashtra)

In [ ]:
if gdf_habitation is not None or gdf_facilities is not None:
    print("\nCleaning the second GeoDataFrame (gdf_habitation)...")
    original_count = len(gdf_habitation)

    # Get the bounding box of each geometry
    bounds = gdf_habitation.geometry.bounds

    # Create a boolean mask to identify valid geometries.
    # This filter will automatically remove the features with the huge negative coordinates.
    is_valid = (bounds['maxy'] <= 90) & (bounds['miny'] >= -90)

    # Overwrite gdf_habitation with the cleaned version
    gdf_habitation = gdf_habitation[is_valid]
    cleaned_count = len(gdf_habitation)
    print(f"Removed {original_count - cleaned_count} features with invalid coordinates.")
    print(f"The cleaned gdf_habitation now has {cleaned_count} features.")

    gdf_facilities = gdf_facilities[is_valid]
    cleaned_count = len(gdf_facilities)
    print(f"Removed {original_count - cleaned_count} features with invalid coordinates.")
    print(f"The cleaned gdf_facilities now has {cleaned_count} features.")

    # This is crucial for any spatial analysis or mapping.
    print("Coordinate Reference System (CRS):")
    print(gdf_facilities.crs)
    print("-" * 30)

    # This is crucial for any spatial analysis or mapping.
    print("Coordinate Reference System (CRS):")
    print(gdf_habitation.crs)
    print("-" * 30)

    print("Coordinate Reference System (CRS):")
    print(gdf_roads.crs)
    print("-" * 30)
    print("Cleaning complete.")

else:
    print("\nOne or all GeoDataFrames are not loaded. Skipping cleaning.")

In [ ]:
def clean_roads_data(gdf_roads):
    if gdf_roads is None:
        print("Roads GeoDataFrame not loaded. Skipping cleaning.")
        return None

    print("\nCleaning roads data...")
    original_count = len(gdf_roads)
    # Extract bounds for each geometry
    bounds = gdf_roads.geometry.bounds
    # Valid lat/lon bounds (WGS84)
    valid_coords = (bounds["minx"] >= -180) & (bounds["maxx"] <= 180) & \
                   (bounds["miny"] >= -90) & (bounds["maxy"] <= 90)

    gdf_roads = gdf_roads[valid_coords].copy()
    print(f"Removed {original_count - len(gdf_roads)} features with invalid coordinates.")

    # Check if any MultiLineString exists
    if (gdf_roads.geometry.type == "MultiLineString").any():
        print("Exploding MultiLineString to LineString...")
        gdf_roads = gdf_roads.explode(index_parts=True).reset_index(drop=True)
        print(f"After exploding: {len(gdf_roads)} features.")


    # Compare geometries (approximate check due to floating-point precision)
    print(f"Removing duplicate roads (if any) original features :{len(gdf_roads)}...")
    gdf_roads["geometry"] = gdf_roads.geometry.round(6)  # Round to 1e-6 degree precision
    gdf_roads = gdf_roads.drop_duplicates(subset=["geometry"])
    print(f"After removing deduplication: {len(gdf_roads)} features.")

    if gdf_roads.crs is None:
        print("Warning: CRS not set. Assuming WGS84 (EPSG:4326).")
        gdf_roads = gdf_roads.set_crs("EPSG:4326")
    else:
        print(f"CRS is set to: {gdf_roads.crs}")

    print("Roads cleaning complete.")
    return gdf_roads

gdf_roads_cleaned = clean_roads_data(gdf_roads)

## Data Cleaning and Validation Code Explanation

This code performs essential data cleaning and validation on geospatial datasets. Here's what it does and why it's important:

### Purpose
The code ensures our geospatial data has valid coordinates before analysis by:
1. Checking if all GeoDataFrames are loaded
2. Removing features with invalid coordinates (outside normal latitude range)
3. Verifying the Coordinate Reference System (CRS) of each dataset

### How It Works

1. **Conditional Execution**:
   - Only runs if at least one GeoDataFrame is loaded (`gdf_habitation`, `gdf_facilities`, or `gdf_roads`)
   - Prevents errors if data isn't available

2. **Coordinate Validation**:
   - Gets bounding box coordinates for each geometry in `gdf_habitation`, `gdf_road`and`gdf_facilities`
   - Creates a boolean mask that identifies valid geometries:
     - Latitude must be ≤ 90° (max valid latitude)
     - Latitude must be ≥ -90° (min valid latitude)
   - This effectively removes features with unrealistic coordinates (like extremely negative values)

3. **Data Filtering**:
   - Applies the filter to both `gdf_habitation` and `gdf_facilities`
   - Tracks and reports how many features were removed
   - Displays the new feature count after cleaning

4. **CRS Verification**:
   - Displays the Coordinate Reference System for each GeoDataFrame
   - Critical for ensuring proper spatial analysis and mapping
   - Incorrect CRS can cause severe distortions in distance calculations and visualizations

### Why This Matters

1. **Data Quality**:
   - Invalid coordinates can cause visualization errors
   - Outliers can skew statistical analyses
   - Ensures all points fall within realistic geographic bounds

2. **Processing Efficiency**:
   - Removes problematic data before computationally expensive operations
   - Prevents potential crashes during spatial operations

3. **Analysis Accuracy**:
   - Ensures distance calculations are meaningful
   - Prevents distorted visualizations
   - Maintains data integrity for spatial queries

This cleaning step is a fundamental best practice in geospatial data processing, especially when working with datasets that may contain errors or inconsistencies in their coordinate values.

In [ ]:
gdf_facilities = gdf_facilities[gdf_facilities['FAC_CATEGO'] == 'Medical'].reset_index(drop=True)

# List of unwanted keywords (animal-related)
unwanted_keywords = ['veterinary hospital', 'animal clinic']

def is_animal_related(text):
    # Check if similarity to any unwanted keyword is above threshold
    for kw in unwanted_keywords:
        if fuzz.partial_ratio(text.lower(), kw.lower()) > 80:
            return True
    return False

# Apply function to filter rows
gdf_facilities['is_animal'] = gdf_facilities['FAC_DESC'].apply(is_animal_related)
gdf_facilities = gdf_facilities[~gdf_facilities['is_animal']].drop(columns='is_animal').reset_index(drop=True)
gdf_facilities

In [ ]:
#export geodataframe to shapefile
gdf_facilities.to_file('/content/drive/MyDrive/Rukiya project/Facilities/Facilities_cleaned.shp')

### Step 1: Broad Categorical Filtering
First, the script performs a broad filter on the dataset. It keeps only the facilities that are explicitly marked with the category **"Medical"**. This step serves as a quick, initial pass to discard a large amount of obviously irrelevant data, such as schools or commercial offices, making the subsequent steps more efficient.

### Step 2: Intelligent Removal of Animal Clinics
The **"Medical"** category is too general and can include animal hospitals. To solve this, the script performs a more sophisticated, fine-grained filtering step:

It uses a technique called fuzzy string matching to intelligently identify veterinary facilities.
Instead of looking for exact text, it compares each facility's description against a list of unwanted keywords (like **"veterinary hospital"** or **"animal clinic"**).
It calculates a **"similarity score"** to catch variations, partial matches, or even minor typos in the facility names.
Any facility whose description is highly similar to one of the animal-related keywords is flagged and removed from the dataset. This method is far more robust and accurate than a simple text search.


In [ ]:
# Define the target Indian CRS
# EPSG:7767 is "Kalianpur 1975 / India zone I", a specific projection
target_crs = "EPSG:7767"

# --- Transform the GeoDataFrames ---

if 'gdf_facilities' in locals() and gdf_facilities is not None:
    print(f"Transforming gdf_facilities to {target_crs}...")
    # The .to_crs() method performs the transformation
    gdf_facilities = gdf_facilities.to_crs(target_crs)
    print(f"New CRS for gdf_facilities: {gdf_facilities.crs.name}")


if 'gdf_habitation' in locals() and gdf_habitation is not None:
    print(f"Transforming gdf_habitation to {target_crs}...")
    gdf_habitation = gdf_habitation.to_crs(target_crs)
    print(f"New CRS for gdf_habitation: {gdf_habitation.crs.name}")


if 'gdf_roads' in locals() and gdf_roads is not None:
     print(f"Transforming gdf_roads to {target_crs}...")
     gdf_roads = gdf_roads.to_crs(target_crs)
     print(f"New CRS for gdf_roads: {gdf_roads.crs.name}")

### The Reason for CRS Conversion (Why is this Necessary?)
#####Geographic CRS vs. Projected CRS

**Geographic CRS (like WGS84, EPSG:4326)**

**What it is**: A system that uses latitude and longitude to define locations on the Earth's 3D, spherical surface.
Units: Degrees (°).

**Best for**: Storing data and pinpointing global locations. This is what GPS uses.
The Problem: Degrees are not a uniform unit of measure. A degree of longitude at the equator is about 111 km wide, but near the poles, it shrinks to almost zero. You cannot accurately measure distance, length, or area using degrees.
Projected CRS (like UTM or EPSG:7767)

**What it is**: A system that mathematically "flattens" the 3D Earth onto a 2D surface, like a paper map. These are always designed to be accurate for a specific region (a "zone").

**Units: Meters or feet.**
**Best for**: Analysis. Because the units are consistent (a meter is a meter everywhere on the map), you can perform accurate calculations.

In [ ]:

if gdf_facilities is not None and gdf_habitation is not None:
    print("\nGenerating the plot with cleaned data...")
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))


    gdf_facilities.plot(ax=axes[0], color='dodgerblue', edgecolor='black')
    axes[0].set_title("Medical Facility of Maharastra")
    axes[0].set_xlabel("Longitude")
    axes[0].set_ylabel("Latitude")
    axes[0].set_aspect('equal', adjustable='box')


    gdf_habitation.plot(ax=axes[1], color='mediumseagreen', edgecolor='black')
    axes[1].set_title("Habitations of Maharashtra")
    axes[1].set_xlabel("Longitude")
    axes[1].set_ylabel("")
    axes[1].set_yticklabels([])
    axes[1].set_aspect('equal', adjustable='box')

    # Print the coordinate reference system (CRS)
    # This is crucial for any spatial analysis or mapping.
    print("Coordinate Reference System (CRS):")
    print(gdf_facilities.crs)
    print("-" * 30)
    # Print the coordinate reference system (CRS)
    # This is crucial for any spatial analysis or mapping.
    print("Coordinate Reference System (CRS):")
    print(gdf_habitation.crs)
    print("-" * 30)
    plt.tight_layout()
    plt.show()
else:
    print("\nPlot generation skipped because one or both files could not be loaded/cleaned.")

In [ ]:
try:
    # GeoPandas has a built-in .plot() method which uses matplotlib.
    print("Generating a simple plot of the shapefile...")
        # Print the coordinate reference system (CRS)
    # This is crucial for any spatial analysis or mapping.
    print("Coordinate Reference System (CRS):")
    print(gdf_facilities.crs)
    print("-" * 30)

    # Create a plot
    fig, ax = plt.subplots(1, 1, figsize=(10, 10))
    gdf_roads.plot(ax=ax)

    # Customize the plot
    ax.set_title("Roads System", fontsize=16)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    plt.show()

except Exception as e:
    print(f"An error occurred: {e}")
    print("\nPlease check the following:")
    print("1. The file path is correct.")
    print("2. The shapefile is not corrupted.")
    print("3. All necessary files (.shp, .shx, .dbf) are present in the directory.")

# **Configurations for the Shortest Path Algorithm**

In [ ]:

_CONFIG = {
    "OUTPUT_BASE_DIR": '/content/drive/MyDrive/network_analysis_output',
    "HABITATION_ID_FIELD": "HAB_ID",
    "FACILITY_ID_FIELD": "FACILITY_I",
    "TARGET_CRS": "EPSG:7767",
    "MAX_CONNECTOR_DISTANCE_METERS": 5000,
    "AOI_BUFFER_METERS": 20000,
    "PROGRESS_UPDATE_INTERVAL_SECONDS": 10,
    "K_NEAREST_FACILITIES_TO_CHECK": 5,
    "ENABLE_INTRA_COMPONENT_FALLBACK": True,
    "ENABLE_MAIN_GRID_FALLBACK": True
}



`"OUTPUT_BASE_DIR"`: This is the base directory where all the output files from the analysis will be saved. In this case, it's set to a specific folder in the Google Drive.

`"HABITATION_ID_FIELD"`: This specifies the name of the column in the habitation dataset that contains the unique identifier for each habitation.

`"FACILITY_ID_FIELD"`: Similarly, this is the name of the column in the facilities dataset that holds the unique ID for each facility.

`"TARGET_CRS"(Just for validation)`: This defines the Coordinate Reference System (CRS) that the data will be transformed into for analysis. **EPSG:7767** is a projected CRS suitable for distance calculations in the region of Maharashtra, India.

`"MAX_CONNECTOR_DISTANCE_METERS"`: This sets the maximum distance (in meters) that a habitation or facility can be from a road segment to be considered "connected" to the road network. Points further away than this distance will not be snapped to the network.

`"AOI_BUFFER_METERS"`: This value determines the buffer distance (in meters) around the combined extent of the habitations and facilities. The road network will be clipped to this buffered area to reduce processing time(the road data had road systems of Gujarat and Goa so this was done to clip that from processing).

`"PROGRESS_UPDATE_INTERVAL_SECONDS"`: This controls how often (in seconds) progress updates are printed to the console during long-running processes like snapping points or computing paths.

`"K_NEAREST_FACILITIES_TO_CHECK"`: For each habitation, the analysis will initially look for the K nearest facilities (based on straight-line distance) as potential destinations. This limits the number of shortest path calculations needed.

`"ENABLE_INTRA_COMPONENT_FALLBACK"`: If enabled (True), the analysis will attempt to find a path to a facility within the same connected component of the road network if a path to the K nearest facilities isn't found.

`"ENABLE_MAIN_GRID_FALLBACK"`: If enabled (True), and if the intra-component fallback fails, the analysis will find the nearest facility anywhere on the largest connected component (the "main grid") of the network, even if a drivable path isn't explicitly found. This provides a fallback assignment but the 'path_type' will indicate a 'jump'(basically this is our Eucladian distances).

`These settings allow us to customize the behavior of the network analysis based on the specific characteristics of your data and the goals of your analysis`

### Helper Class: Progress Tracker

In [ ]:

class ProgressTracker:
    def __init__(self, total_items, interval_seconds=10):
        self.total = total_items
        self.interval = interval_seconds
        self.start_time = time.time()
        self.last_update_time = self.start_time
        self.item_name = "items"
    def set_item_name(self, name):
        self.item_name = name
        return self
    def check(self, current_index):
        now = time.time()
        if now - self.last_update_time > self.interval:
            elapsed = now - self.start_time
            percent_complete = ((current_index + 1) / self.total) * 100
            print(f"      - Progress: {current_index + 1}/{self.total} {self.item_name} processed ({percent_complete:.1f}%). Total time: {elapsed:.1f}s")
            self.last_update_time = now



This section defines a simple `ProgressTracker` class. Its purpose is to provide visual feedback during long-running loops or operations within the analysis. It prints periodic updates to the console showing the current progress (items processed out of total) and the elapsed time, helping to monitor the execution flow.

### Analysis Functions

In [ ]:

def find_k_nearest_candidates(gdf_left, gdf_right, k, left_id_field, right_id_field):
    print("  Calculating K-nearest candidates using cKDTree...")
    right_coords = np.array([pt.coords[0] for pt in gdf_right.geometry])
    left_coords = np.array([pt.coords[0] for pt in gdf_left.geometry])
    tree = cKDTree(right_coords)
    distances, indices = tree.query(left_coords, k=k)
    candidate_map = {}
    left_ids = gdf_left[left_id_field].values
    right_ids = gdf_right[right_id_field].values
    if k == 1:
        for i, left_id in enumerate(left_ids):
            candidate_map[left_id] = [right_ids[indices[i]]]
    else:
        for i, left_id in enumerate(left_ids):
            candidate_map[left_id] = right_ids[indices[i]].tolist()
    return candidate_map

def create_graph_from_gdf(gdf_roads):
    net = nx.Graph()
    gdf_roads = gdf_roads.reset_index(drop=True).rename_axis('road_idx').reset_index()
    for row in gdf_roads.itertuples(index=False):
        line = row.geometry
        if isinstance(line, LineString) and not line.is_empty and line.geom_type == 'LineString':
            length = line.length
            start_node, end_node = line.coords[0], line.coords[-1]
            net.add_edge(start_node, end_node, weight=length, road_idx=row.road_idx)
    return net, gdf_roads

def snap_and_split_algo(points_gdf, roads_gdf, points_id_field, max_dist):
    print(f" - Finding snap points for {len(points_gdf)} points...")
    if not hasattr(roads_gdf, 'sindex'):
        print("    - Building spatial index for roads...")
        roads_gdf.sindex
    entry_points = {}
    points_to_split_by_road_idx = defaultdict(list)
    tracker = ProgressTracker(len(points_gdf), _CONFIG["PROGRESS_UPDATE_INTERVAL_SECONDS"]).set_item_name("points")
    for i, (point_id, point_geom) in enumerate(zip(points_gdf[points_id_field], points_gdf.geometry)):
        tracker.check(i)
        if point_geom is None or point_geom.is_empty: continue
        try:
            possible_matches_indices = list(roads_gdf.sindex.nearest(point_geom, return_all=False)[1])
            if not possible_matches_indices: continue
            nearest_road_series = roads_gdf.iloc[possible_matches_indices[0]]
            nearest_road_geom = nearest_road_series.geometry
            nearest_road_index = nearest_road_series.name
            snapped_point = nearest_points(point_geom, nearest_road_geom)[1]
            connector_dist = point_geom.distance(snapped_point)
            if connector_dist <= max_dist:
                entry_points[point_id] = {"node": snapped_point.coords[0], "connector_dist": connector_dist, "connector_geom": LineString([point_geom, snapped_point])}
                points_to_split_by_road_idx[nearest_road_index].append(snapped_point)
        except Exception:
            continue
    print(f"    - Snapping complete. {len(points_to_split_by_road_idx)} roads will be split.")
    newly_split_geoms = []
    road_indices_to_drop = set(points_to_split_by_road_idx.keys())
    print("    - Splitting affected road segments...")
    for road_idx, split_points in points_to_split_by_road_idx.items():
        road_geom = roads_gdf.loc[road_idx, 'geometry']
        splitter = MultiPoint(split_points)
        try:
            split_geoms = split(snap(road_geom, splitter, 1e-9), splitter)
            for geom in split_geoms.geoms:
                if isinstance(geom, LineString) and geom.length > 1e-6:
                    newly_split_geoms.append(geom)
        except Exception:
            newly_split_geoms.append(road_geom)
    untouched_roads_gdf = roads_gdf.drop(index=list(road_indices_to_drop))
    new_roads_gdf = gpd.GeoDataFrame(geometry=newly_split_geoms, crs=roads_gdf.crs)
    final_roads_gdf = pd.concat([untouched_roads_gdf, new_roads_gdf], ignore_index=True)
    print(f"    - New road network created with {len(final_roads_gdf)} segments.")
    return final_roads_gdf, entry_points

def to_multilinestring(geom):
    if isinstance(geom, list):
        valid_geoms = [g for g in geom if g and not g.is_empty]
        if not valid_geoms:
            return None
        geom = linemerge(valid_geoms)
    if geom is None or geom.is_empty:
        return None
    if geom.geom_type == 'LineString':
        return MultiLineString([geom])
    elif geom.geom_type == 'MultiLineString':
        return geom
    return None


def process_full_network(analysis_name, gdf_habitations_proj, gdf_facilities_proj, gdf_roads_proj, gdf_habs_orig_crs):
    try:
        print(f"\n--- Processing: {analysis_name} ---")
        hab_id_field = _CONFIG["HABITATION_ID_FIELD"]
        fac_id_field = _CONFIG["FACILITY_ID_FIELD"]

        # Define the main output directory for this analysis run
        output_dir = os.path.join(_CONFIG["OUTPUT_BASE_DIR"], analysis_name)
        os.makedirs(output_dir, exist_ok=True)

        gdf_habs_cleaned = gdf_habitations_proj[[hab_id_field, 'geometry']].dropna().copy()
        gdf_facs_cleaned = gdf_facilities_proj[[fac_id_field, 'geometry']].dropna().copy()
        gdf_roads_cleaned = gdf_roads_proj.explode(ignore_index=True).reset_index(drop=True)
        gdf_roads_cleaned = gdf_roads_cleaned.loc[gdf_roads_cleaned.geometry.geom_type == 'LineString', ['geometry']].copy()
        del gdf_roads_proj

        if gdf_roads_cleaned.empty or gdf_habs_cleaned.empty or gdf_facs_cleaned.empty: return "Skipped: No valid geometries."

        print(f"  - Stage 1: Finding K-nearest facility candidates...")
        hab_to_fac_candidates = find_k_nearest_candidates(gdf_habs_cleaned, gdf_facs_cleaned, k=_CONFIG['K_NEAREST_FACILITIES_TO_CHECK'], left_id_field=hab_id_field, right_id_field=fac_id_field)

        print("  - Stage 2: Snapping points and splitting road network...")
        habs_split_roads, hab_entry_points = snap_and_split_algo(gdf_habs_cleaned, gdf_roads_cleaned, hab_id_field, _CONFIG["MAX_CONNECTOR_DISTANCE_METERS"])
        del gdf_roads_cleaned
        final_split_roads_unindexed, fac_entry_points = snap_and_split_algo(gdf_facs_cleaned, habs_split_roads, fac_id_field, _CONFIG["MAX_CONNECTOR_DISTANCE_METERS"])
        del habs_split_roads

        print("  - Stage 3: Building memory-optimized graph...")
        road_network, final_split_roads = create_graph_from_gdf(final_split_roads_unindexed)
        del final_split_roads_unindexed

        # --- ADD THIS CODE BLOCK ---
        print("  - Stage 3.1: Exporting the cleaned and split road network...")
        cleaned_roads_output_path = os.path.join(output_dir, "cleaned_roads/cleaned_and_split_roads.shp")
        final_split_roads.to_file(cleaned_roads_output_path)
        print(f"    - Cleaned road network saved to: {cleaned_roads_output_path}")
        # --- END OF ADDED CODE ---

        path_info_for_habs = {}

        if road_network.number_of_nodes() > 0:
            print("  - Stage 3.5: Analyzing network components for robust fallbacks...")
            node_to_component_id = {}
            component_id_to_fac_nodes = defaultdict(list)
            fac_node_to_id_map = {v['node']: k for k, v in fac_entry_points.items()}

            connected_components = list(nx.connected_components(road_network))
            main_grid_component_id = -1
            if connected_components:
                main_grid_component = max(connected_components, key=len)
                print(f"    - Main grid identified with {len(main_grid_component)} nodes.")
                for i, component in enumerate(connected_components):
                    if component == main_grid_component: main_grid_component_id = i
                    for node in component:
                        node_to_component_id[node] = i
                        if node in fac_node_to_id_map: component_id_to_fac_nodes[i].append(node)

            main_grid_facility_nodes = component_id_to_fac_nodes.get(main_grid_component_id, [])
            print(f"    - Found {len(component_id_to_fac_nodes)} components with facilities.")
            print(f"    - Found {len(main_grid_facility_nodes)} facilities on the main grid.")

            print("  - Stage 4: Pre-computing path LENGTHS from FACILITY nodes...")
            precomputed_lengths = {}
            unique_fac_nodes = {v['node'] for v in fac_entry_points.values() if v['node'] in road_network}
            tracker = ProgressTracker(len(unique_fac_nodes), _CONFIG["PROGRESS_UPDATE_INTERVAL_SECONDS"]).set_item_name("source facility nodes")
            for i, fac_node in enumerate(unique_fac_nodes):
                tracker.check(i)
                precomputed_lengths[fac_node] = nx.single_source_dijkstra_path_length(road_network, source=fac_node, weight='weight')
            print("    - Path length pre-computation complete.")

            print("  - Stage 4.5: Pre-building spatial index for Main Grid Fallback...")
            main_grid_kdtree = None
            main_grid_fac_nodes_list = []
            if _CONFIG["ENABLE_MAIN_GRID_FALLBACK"] and main_grid_facility_nodes:
                main_grid_fac_nodes_list = list(main_grid_facility_nodes)
                main_grid_facility_coords = np.array(main_grid_fac_nodes_list)
                main_grid_kdtree = cKDTree(main_grid_facility_coords)
                print(f"    - cKDTree built for {len(main_grid_fac_nodes_list)} main grid facilities.")

            print("  - Stage 5: Assigning facilities using multi-tiered logic...")
            tracker = ProgressTracker(len(hab_to_fac_candidates), _CONFIG["PROGRESS_UPDATE_INTERVAL_SECONDS"]).set_item_name("habitations")

            for i, (hab_id, candidate_fac_ids) in enumerate(hab_to_fac_candidates.items()):
                tracker.check(i)
                if hab_id not in hab_entry_points: continue

                hab_node = hab_entry_points[hab_id]['node']
                best_fac_id, min_dist, reason, path_nodes = None, float('inf'), None, None

                for fac_id in candidate_fac_ids:
                    if fac_id not in fac_entry_points: continue
                    fac_node = fac_entry_points[fac_id]['node']
                    if fac_node in precomputed_lengths:
                        road_dist = precomputed_lengths[fac_node].get(hab_node)
                        if road_dist is not None:
                            total_dist = hab_entry_points[hab_id]['connector_dist'] + road_dist + fac_entry_points[fac_id]['connector_dist']
                            if total_dist < min_dist: min_dist, best_fac_id, reason = total_dist, fac_id, 'Nearest Found'

                if not best_fac_id and _CONFIG["ENABLE_INTRA_COMPONENT_FALLBACK"] and hab_node in node_to_component_id:
                    hab_component_id = node_to_component_id[hab_node]
                    if hab_component_id in component_id_to_fac_nodes:
                        for fac_node_in_component in component_id_to_fac_nodes[hab_component_id]:
                            road_dist = precomputed_lengths[fac_node_in_component].get(hab_node)
                            if road_dist is not None:
                                fac_id = fac_node_to_id_map[fac_node_in_component]
                                total_dist = hab_entry_points[hab_id]['connector_dist'] + road_dist + fac_entry_points[fac_id]['connector_dist']
                                if total_dist < min_dist: min_dist, best_fac_id, reason = total_dist, fac_id, 'Intra-Component Fallback'

                if not best_fac_id and main_grid_kdtree:
                    distance, index = main_grid_kdtree.query(hab_node)
                    target_fac_node_on_main_grid = main_grid_fac_nodes_list[index]
                    best_fac_id = fac_node_to_id_map[target_fac_node_on_main_grid]
                    total_dist = hab_entry_points[hab_id]['connector_dist'] + distance + fac_entry_points[best_fac_id]['connector_dist']
                    min_dist, reason = total_dist, 'Main Grid Fallback'


                if best_fac_id:

                    if reason in ['Nearest Found', 'Intra-Component Fallback']:
                        final_fac_node = fac_entry_points[best_fac_id]['node']
                        try:
                            path_nodes = nx.shortest_path(road_network, source=hab_node, target=final_fac_node, weight='weight')
                        except (nx.NetworkXNoPath, nx.NodeNotFound):
                            path_nodes = None

                    if path_nodes or reason == 'Main Grid Fallback':
                         path_info_for_habs[hab_id] = {'found': True, 'fac_id': best_fac_id, 'total_dist': min_dist, 'reason': reason, 'path_nodes': path_nodes}

        print("  - Stage 6: Identifying unconnected habitations...")
        unconnected_hab_ids = [row[hab_id_field] for _, row in gdf_habs_cleaned.iterrows() if row[hab_id_field] not in path_info_for_habs]
        if unconnected_hab_ids:
            print(f"    - Found {len(unconnected_hab_ids)} habitations that could not be connected via the network. They will be marked as 'No Path Found'.")
        else:
            print("    - All habitations were successfully connected via the network.")

        print("  - Stage 7: Preparing final files...")
        successful_paths = []
        for hab_id, path_data in path_info_for_habs.items():
            if not path_data.get('found'): continue
            hab_info, fac_info = hab_entry_points.get(hab_id), fac_entry_points.get(path_data['fac_id'])
            reason, full_path_geom = path_data.get('reason'), None

            if not hab_info or not fac_info: continue

            if reason in ['Nearest Found', 'Intra-Component Fallback']:
                path_nodes = path_data['path_nodes']
                if path_nodes:
                    road_indices = [road_network.edges[u, v]['road_idx'] for u, v in zip(path_nodes[:-1], path_nodes[1:])]
                    road_geoms = final_split_roads.loc[road_indices, 'geometry'].tolist()
                    full_path_geom = [hab_info['connector_geom'], linemerge(road_geoms), fac_info['connector_geom']]

            elif reason == 'Main Grid Fallback':
                full_path_geom = [hab_info['connector_geom'], fac_info['connector_geom']]

            multi_line_geom = to_multilinestring(full_path_geom)
            if multi_line_geom:
                successful_paths.append({"HAB_ID": hab_id, "FACILITY_I": path_data['fac_id'], "path_len_m": path_data['total_dist'], "path_type": reason, "geometry": multi_line_geom})

        hab_results = []
        for _, hab_row in gdf_habs_orig_crs.iterrows():
            hab_id = hab_row.get(hab_id_field)
            info = path_info_for_habs.get(hab_id)
            reason_no_path, path_found = '', bool(info and info.get('found'))
            if not path_found:
                reason_no_path = 'No Path Found'
                if hab_id not in hab_entry_points: reason_no_path = f"Habitation >{_CONFIG['MAX_CONNECTOR_DISTANCE_METERS']}m from road"

            hab_results.append({"HAB_ID": hab_id, "path_found": path_found, "nearest_fac_id": info['fac_id'] if info else None, "path_len_m": info['total_dist'] if info else None, "path_type": info.get('reason') if info else None, "reason_no_path": reason_no_path, "geometry": hab_row.geometry})

        gpd.GeoDataFrame(hab_results, crs=gdf_habs_orig_crs.crs).to_file(os.path.join(output_dir, "habitations_with_path_info.shp"))
        if not successful_paths: return f"Warning: Analysis complete, but no valid paths were found."
        gpd.GeoDataFrame(successful_paths, crs=_CONFIG['TARGET_CRS']).to_file(os.path.join(output_dir, "shortest_paths.shp"))

        return f"Success: {len(successful_paths)} paths found. Results saved to {output_dir}"

    except Exception as e:
        return f"ERROR processing {analysis_name}:\n{traceback.format_exc()}"



This block contains the core functions implementing the network analysis logic. These functions are designed for performance and handle specific tasks:

*   `find_k_nearest_candidates`: Efficiently identifies the K nearest potential facilities for each habitation using a K-D tree spatial index.
*   `create_graph_from_gdf`: Builds a NetworkX graph representation of the road network GeoDataFrame, which is essential for shortest path calculations.
*   `snap_and_split_algo`: Snaps point locations (habitations and facilities) to the nearest road segments within a defined distance and splits the road geometries at these snap points. This step is crucial for connecting the points to the network graph accurately.
*   `to_multilinestring`: A helper function to ensure geometries are in the correct MultiLineString format for output.
*   `process_full_network`: The main function that orchestrates the entire analysis pipeline: candidate finding, snapping, graph building, shortest path calculation, handling different connection scenarios, and saving the results.

### Main Execution Block - Pre-processing

In [ ]:

if __name__ == '__main__':
    total_start_time = time.time()
    print("\n--- Starting Full Network Analysis ---")
    try:
        print("Checking for required GeoDataFrames...")
        if 'gdf_facilities' not in locals() or 'gdf_habitation' not in locals() or 'gdf_roads' not in locals():
             raise NameError("Required GeoDataFrames (gdf_facilities, gdf_habitation, gdf_roads) are not defined.")
        print("All required GeoDataFrames are available.")
    except NameError as e:
        print(f"\nFATAL ERROR: {e}")
        print("Please ensure your data is loaded into the correct variables before running this script.")
    else:
        print("\n--- Pre-processing: Spatially filtering large road network ---")
        target_crs = _CONFIG['TARGET_CRS']
        gdf_facilities_proj = gdf_facilities.to_crs(target_crs)
        gdf_habitations_proj = gdf_habitation.to_crs(target_crs)
        gdf_roads_proj_full = gdf_roads.to_crs(target_crs)

        print("  - Defining Area of Interest (AOI) from combined point data...")
        combined_points_geoms = pd.concat([gdf_facilities_proj['geometry'], gdf_habitations_proj['geometry']], ignore_index=True)
        aoi_poly = combined_points_geoms.unary_union.envelope
        aoi_buffer_poly = gpd.GeoDataFrame(geometry=[aoi_poly.buffer(_CONFIG['AOI_BUFFER_METERS'])], crs=target_crs)
        print(f"  - AOI defined and buffered by {_CONFIG['AOI_BUFFER_METERS']} meters.")

        print("  - Clipping the national road network. This may take a moment...")
        roads_clipped = gpd.clip(gdf_roads_proj_full, aoi_buffer_poly)
        print(f"  - Clipping complete. Road segments reduced from {len(gdf_roads_proj_full)} to {len(roads_clipped)}.")

        del gdf_roads_proj_full, aoi_buffer_poly, combined_points_geoms



This section sets up the environment and performs the necessary pre-processing steps before calling the main analysis function. This includes:

1.  **Timing**: Recording the start time to measure the total execution duration.
2.  **Dependency Check**: Verifying that the necessary GeoDataFrames (`gdf_facilities`, `gdf_habitation`, `gdf_roads`) are available in the environment.
3.  **CRS Transformation**: Transforming the input GeoDataFrames to the target CRS (`EPSG:7767`) for accurate spatial analysis.
4.  **Area of Interest (AOI) Definition**: Creating a spatial boundary (buffer around the combined extent of facilities and habitations) to focus the road network clipping.
5.  **Road Network Clipping**: Reducing the size of the extensive national road network by clipping it to the defined AOI. This significantly improves processing performance by reducing the number of road segments considered in the graph building and shortest path calculations.
6.  **Memory Management**: Deleting large intermediate GeoDataFrames (`gdf_roads_proj_full`, `aoi_buffer_poly`, `combined_points_geoms`) to free up memory.

In [ ]:

result = process_full_network(
    "Maharashtra_Analysis",
    gdf_habitations_proj,
    gdf_facilities_proj,
    roads_clipped,
    gdf_habitation
)
print("\n\n--- Analysis Complete ---")
print(f"Result: {result}")
total_execution_time = time.time() - total_start_time
print(f"\nTotal execution time: {total_execution_time / 60:.2f} minutes.")
print(f"All outputs saved in: {_CONFIG['OUTPUT_BASE_DIR']}")

# **Configrations for Service level Analysis**

In [ ]:
_CONFIG = {

    "habitation_shapefile_path": r'/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis_No_Jump_Lines/shortest_paths.shp',
    "PATH_INFO_SHAPEFILE_PATH": r'/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis_No_Jump_Lines/habitations_with_path_info.shp',
    "OUTPUT_DIR": r'/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis/ServiceLevelAnalysis',
    "output_filename": "maharashtra_habitation_service_levels.shp",
    "HABITATION_ID_FIELD": "HAB_ID",
    "SERVICE_DISTANCE_METERS": 5000
}

`"habitation_shapefile_path"`: This is the path to the shapefile containing the shortest path geometries generated by the previous network analysis step. It's where the lines representing the calculated routes from habitations to facilities are stored. Note: While this variable name mentions "habitation_shapefile_path", the path points to the shortest_paths.shp file, which contains the lines, not the habitation points.

`"PATH_INFO_SHAPEFILE_PATH"`: This is the path to another shapefile output from the previous network analysis. This file is crucial because it contains the original habitation points plus additional information calculated during the analysis, such as whether a path was found, the ID of the nearest facility, and the calculated path length (path_len_m). This is the main input for the service level classification.

`"OUTPUT_DIR"`: This specifies the directory where the results of the service level analysis will be saved.

`"output_filename"`: This is the name that will be given to the output shapefile generated by this service level analysis.

`"HABITATION_ID_FIELD"`: This confirms the name of the column used to uniquely identify habitations, ensuring consistency with the previous analysis steps.

`"SERVICE_DISTANCE_METERS"`: This is a key parameter for the service level analysis. It defines the threshold distance (in meters). Habitations with a calculated path length to a medical facility less than or equal to this value will be considered "Served", while those with a path length greater than this value (or no path found) will be considered "Underserved".

`These settings allow us to customize the behavior of the network analysis based on the specific characteristics of your data and the goals of your analysis`



## About classify_habitation function

In [ ]:
def classify_habitation(row, distance_threshold):
    path_length = row['path_len_m']

    if pd.isna(path_length) or path_length > distance_threshold:
        return "Underserved"
    else:
        return "Served"

### This function takes two arguments

`row`: This is expected to be a single row from a pandas DataFrame or GeoDataFrame.

`distance_threshold`: This is a numerical value representing the maximum distance (in meters, based on the configuration in the previous cell) for a habitation to be considered "Served".

### The function works as follows:

It first retrieves the value from the `'path_len_m'` column for the given row. This column is assumed to contain the calculated shortest path length from the habitation to its nearest medical facility.

### It then checks two conditions:

`pd.isna(path_length)`: This checks if the path_length is a missing or "Not a Number" (NaN) value. This would typically happen if no path was found for this habitation during the network analysis.
path_length > distance_threshold: This checks if the calculated path_length is greater than the distance_threshold provided.
If either of these conditions is true (meaning the path length is missing or it's longer than the allowed threshold), the function returns the string "Underserved".
If neither condition is true (meaning a path was found and its length is less than or equal to the threshold), the function returns the string "Served".

In [ ]:
try:
    total_start_time = time.time()
    print("--- Starting Service Level Analysis for Maharashtra ---")
    print("  - Loading analysis results and original habitations...")

    if not os.path.exists(_CONFIG["PATH_INFO_SHAPEFILE_PATH"]):
        print(f"FATAL ERROR: Path info file not found at '{_CONFIG['PATH_INFO_SHAPEFILE_PATH']}'")
    elif not os.path.exists(_CONFIG["habitation_shapefile_path"]):
        print(f"FATAL ERROR: Original habitation file not found at '{_CONFIG['habitation_shapefile_path']}'")
    else:
        gdf_habs_with_info = gpd.read_file(_CONFIG["PATH_INFO_SHAPEFILE_PATH"])

        if gdf_habs_with_info.empty:
            print("FATAL ERROR: Path info file is empty.")
        else:
            print(f"  - Classifying {len(gdf_habs_with_info)} habitations based on {_CONFIG['SERVICE_DISTANCE_METERS']}m threshold...")


            gdf_habs_with_info['service_level'] = gdf_habs_with_info.apply(
                classify_habitation,
                distance_threshold=_CONFIG["SERVICE_DISTANCE_METERS"],
                axis=1
            )
            print("  - Saving classified habitation points...")
            output_columns = [
                _CONFIG["HABITATION_ID_FIELD"],
                'geometry',
                'path_len_m',
                'service_level',
                'nearest_fa',
                'reason_no_',
                'path_type'
            ]

            final_columns = [col for col in output_columns if col in gdf_habs_with_info.columns]
            gdf_final = gdf_habs_with_info[final_columns]

            os.makedirs(_CONFIG["OUTPUT_DIR"], exist_ok=True)
            output_path = os.path.join(_CONFIG["OUTPUT_DIR"], _CONFIG["output_filename"])

            gdf_final.to_file(output_path, driver="ESRI Shapefile")
            service_counts = gdf_final['service_level'].value_counts().to_dict()
            served_count = service_counts.get("Served", 0)
            underserved_count = service_counts.get("Underserved", 0)

            print("\n--- Analysis Complete ---")
            print(f"Successfully classified {len(gdf_final)} habitations.")
            print(f"  - Served: {served_count}")
            print(f"  - Underserved: {underserved_count}")
            print(f"Final shapefile saved to: {output_path}")

except Exception as e:
    print(f"\n--- An ERROR occurred during processing ---")
    tb_str = traceback.format_exc()
    print(tb_str)

finally:
    total_execution_time = time.time() - total_start_time
    print(f"\nTotal execution time: {total_execution_time:.2f} seconds.")

### Here's a breakdown of the code:

`Error Handling (try...except...finally)`: The entire block is wrapped in a try...except...finally structure. This is good practice for robust code.
The try block contains the main logic. If any error occurs within this block, the code jumps to the except block.

The except Exception as e: block catches any type of exception that might occur, prints an error message, and displays the traceback to help diagnose the issue.

`The finally`: block is guaranteed to execute regardless of whether an error occurred or not. In this case, it's used to calculate and print the total execution time of the analysis.

###Start Time and Initial Output:

`total_start_time = time.time()`: Records the starting time of the analysis to measure how long it takes.
print("--- Starting Service Level Analysis for Maharashtra ---"): Prints a clear message indicating the start of this process.

###Load Required Data:
print(" - Loading analysis results and original habitations..."): Indicates the data loading step.
The if not os.path.exists(...) checks ensure that the necessary input files (habitations_with_path_info.shp) exist before attempting to read them. If they are missing, it prints a fatal error message.
gdf_habs_with_info = gpd.read_file(_CONFIG["PATH_INFO_SHAPEFILE_PATH"]): Reads the shapefile containing the habitation data along with the network analysis results (like path_len_m, nearest_fac_id, etc.) into a GeoDataFrame.
if gdf_habs_with_info.empty:: Checks if the loaded GeoDataFrame is empty and prints a fatal error if it is.

###Classify Habitations:
`print(f" - Classifying {len(gdf_habs_with_info)} habitations based on {_CONFIG['SERVICE_DISTANCE_METERS']}m threshold...")`: Reports the number of habitations being classified and the distance threshold being used.

`gdf_habs_with_info['service_level'] = gdf_habs_with_info.apply(classify_habitation, distance_threshold=_CONFIG["SERVICE_DISTANCE_METERS"], axis=1)`: This is the core classification step. It uses the .apply() method on the GeoDataFrame to call the classify_habitation function (defined in the previous cell) for each row (axis=1). The result of the classify_habitation function ("Served" or "Underserved") is assigned to a new column named 'service_level' in the GeoDataFrame.

###Prepare and Save Final Output:

print(" - Saving classified habitation points..."): Indicates the saving step.

output_columns = [...]: Defines a list of column names to include in the final output shapefile. It specifically includes the potentially truncated names ('nearest_fa', 'reason_no_') that might result from saving to the ESRI Shapefile format (which has limitations on column name length).
final_columns = [col for col in output_columns if col in gdf_habs_with_info.columns]: Creates a list of columns that actually exist in the gdf_habs_with_info GeoDataFrame, ensuring that only existing columns are selected.

gdf_final = gdf_habs_with_info[final_columns]: Selects the specified columns to create the final GeoDataFrame for saving.
os.makedirs(_CONFIG["OUTPUT_DIR"], exist_ok=True): Creates the output directory if it doesn't already exist.
output_path = os.path.join(_CONFIG["OUTPUT_DIR"], _CONFIG["output_filename"]): Constructs the full path for the output shapefile.
gdf_final.to_file(output_path, driver="ESRI Shapefile"): Saves the gdf_final GeoDataFrame as a shapefile. The driver="ESRI Shapefile" specifies the file format.

###Summarize Results:
`service_counts = gdf_final['service_level'].value_counts().to_dict()`: Counts how many habitations fall into each category ("Served" and "Underserved").
Prints a summary of the analysis, including the total number of habitations classified and the counts for "Served" and "Underserved".
Prints the path where the final shapefile was saved.

###Final Execution Time:
The finally block calculates the total execution time and prints it.
In essence, this code orchestrates the service level analysis by loading the necessary data, applying the classification logic defined in the classify_habitation function, and saving the results to a new shapefile, providing a summary of the classification counts.



# **Configuration for the Clustering and Suggestion of New Medical Facilities**

In [ ]:
_CONFIG = {
    "INPUT_SHAPEFILE": r'/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis_No_Jump_Lines/ServiceLevelAnalysis/maharashtra_habitation_service_levels.shp',
    "ROAD_NETWORK_SHAPEFILE": r'/content/drive/MyDrive/Rukiya project/Roads/gis_osm_roads_free_1.shp',
    "OUTPUT_DIR": r'/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis/ServiceLevelAnalysis/HierarchicalClustering/',
    "TARGET_CRS": "EPSG:7767",
    "SERVICE_LEVEL_FIELD": "service_le",
    "HABITATION_ID_FIELD": "HAB_ID",
    "DBSCAN_EPS_METERS": 5000,
    "DBSCAN_MIN_SAMPLES": 10,
    "HABS_PER_FACILITY_THRESHOLD": 250,
    "MEDOID_CANDIDATE_SAMPLE_SIZE": 20,
    "HYBRID_NETWORK_FILENAME": "hierarchical_analysis_network.shp",
    "HABITATIONS_FOLDER": "Habitations_Output",
    "FACILITIES_FOLDER": "Facilities_Output",
    "NETWORK_FOLDER": "Network_Output",
    "ORIGINAL_FACILITIES_SHAPEFILE": r'/content/drive/MyDrive/Rukiya project/Facilities/Facilities_cleaned.shp',
    "HABS_COLUMNS_TO_KEEP": ["HAB_ID", "service_le", "path_len_m", "geometry"],
    "ROADS_COLUMNS_TO_KEEP": ["geometry"],
    "FACILITIES_COLUMNS_TO_KEEP": ["geometry"],
    "MIN_ROAD_SEGMENT_LENGTH": 10.0,
    "SILHOUETTE_SAMPLE_SIZE": 2500
}

### 1. Data I/O Parameters
This section defines the locations of input source data and the destination for all generated artifacts.

#### INPUT_SHAPEFILE:

Description: Specifies the path to the primary input shapefile containing habitation point data for Maharashtra.
Role: This dataset serves as the foundational layer for the analysis, with each feature representing a habitation and its associated attributes.

#### ROAD_NETWORK_SHAPEFILE:

Description: The path to the shapefile representing the regional road network.
Role: Provides the underlying graph structure for network analysis, routing, and connectivity assessments.

#### OUTPUT_DIR:

Description: The primary directory where all analysis outputs will be stored.
Role: Centralizes all generated files, including shapefiles, reports, and visualizations, ensuring an organized project structure.

#### HYBRID_NETWORK_FILENAME, HABITATIONS_FOLDER, etc.:

Description: Defines the naming convention and sub-directory structure for specific output categories.
Role: Enforces a consistent and logical organization of output artifacts within the main OUTPUT_DIR.

#### 2. Geospatial and Attribute Specifications
These parameters ensure spatial consistency and correct data interpretation.

#### TARGET_CRS:

Description: Defines the target Coordinate Reference System (CRS) as "EPSG:7767".
Role: This is a critical parameter. EPSG:7767 is a projected CRS using meters as its unit. All geospatial data is transformed into this CRS to enable accurate and meaningful distance, buffer, and density calculations, which are unreliable when using a geographic CRS (e.g., WGS84).

#### SERVICE_LEVEL_FIELD:

Description: The attribute field name (service_le) within the INPUT_SHAPEFILE that contains the service level classification for each habitation.
Role: Allows the script to dynamically access the target variable for analysis without hardcoding the column name.

#### HABITATION_ID_FIELD:

Description: The attribute field name (HAB_ID) serving as the unique identifier for each habitation.
Role: Ensures proper indexing, joining, and tracking of individual habitation features throughout the analysis.

### 3. Analysis & Clustering Hyperparameters
This section configures the core algorithm used for spatial clustering: Density-Based Spatial Clustering of Applications with Noise (DBSCAN).

#### DBSCAN_EPS_METERS:

Description: Sets the epsilon (ε) hyperparameter for DBSCAN to 5000 meters.
Role: Defines the search radius around each point to find neighbors. This value directly controls the spatial extent of what is considered a "local neighborhood" for cluster formation.

#### DBSCAN_MIN_SAMPLES:

Description: Sets the min_samples hyperparameter to 10.
Role: Specifies the minimum number of points required within a point's eps radius (including the point itself) for it to be classified as a "core point" and initiate a cluster. This parameter effectively defines the density threshold required to form a valid cluster, thereby filtering out sparse, noisy data points.

`These settings allow us to customize the behavior of the network analysis based on the specific characteristics of your data and the goals of your analysis`

## **Analysis Pipeline Functions: Detailed Breakdown**

In [ ]:
def load_and_prepare_data(filepath, service_level_field):
    """Loads and filters the main habitations shapefile."""
    print("  - Loading classified habitations shapefile...")
    gdf = gpd.read_file(filepath)
    gdf.rename(columns={service_level_field: 'service_level'}, inplace=True)
    print("  - Filtering for 'Underserved' habitations...")
    gdf_underserved = gdf[gdf['service_level'] == 'Underserved'].copy()
    if len(gdf_underserved) == 0: return None
    print(f"  - Found {len(gdf_underserved)} underserved habitations to analyze.")
    return gdf_underserved


def perform_dbscan_clustering(gdf, eps, min_samples):
    """Runs the initial, standard (unweighted) DBSCAN to find large regions of need."""
    print(f"  - Running STANDARD (unweighted) DBSCAN with eps={eps}m and min_samples={min_samples}...")
    coords = np.array(list(gdf.geometry.apply(lambda p: (p.x, p.y))))
    db = DBSCAN(eps=eps, min_samples=min_samples, algorithm='ball_tree').fit(coords)
    gdf.loc[:, 'cluster_id'] = db.labels_

    n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
    n_outliers = list(db.labels_).count(-1)
    print(f"  - DBSCAN identified {n_clusters} large 'regions of need'.")
    print(f"  - Found {n_outliers} habitations classified as outliers.")
    return gdf, n_clusters, n_outliers


def find_network_medoid(sub_cluster_group, full_graph, nodes_gdf, candidate_sample_size):
    """Finds the single best Network Medoid for a given group of habitations."""
    hab_geometries = sub_cluster_group.geometry
    minx, miny, maxx, maxy = hab_geometries.total_bounds
    candidate_nodes_gdf = nodes_gdf.cx[minx:maxx, miny:maxy]

    if len(candidate_nodes_gdf) == 0:
        center = sub_cluster_group.geometry.union_all().centroid
        return nodes_gdf.iloc[[nodes_gdf.geometry.distance(center).idxmin()]]

    if len(candidate_nodes_gdf) > candidate_sample_size:
        center = sub_cluster_group.geometry.union_all().centroid
        closest_indices = candidate_nodes_gdf.geometry.distance(center).nsmallest(candidate_sample_size).index
        candidate_nodes_gdf = candidate_nodes_gdf.loc[closest_indices]

    hab_node_ids = sub_cluster_group['nodeID'].tolist()
    candidate_node_ids = candidate_nodes_gdf['nodeID'].tolist()
    best_candidate_id, min_total_distance = -1, float('inf')

    for cid in candidate_node_ids:
        try:
            lengths = nx.single_source_dijkstra_path_length(full_graph, cid, weight='length')
            dist = sum(lengths.get(hid, float('inf')) for hid in hab_node_ids)
            if dist < min_total_distance and dist != 0:
                min_total_distance = dist
                best_candidate_id = cid
        except (nx.NodeNotFound, nx.NetworkXNoPath):
            continue

    if best_candidate_id == -1: return None
    return nodes_gdf[nodes_gdf['nodeID'] == best_candidate_id]

def hierarchical_facility_placement(gdf_clustered, hab_id_field, full_graph, nodes_gdf, config):
    """
    Implements a hierarchical placement strategy with data-driven sub-clustering (Elbow Method).
    This version is CORRECTED to handle column naming from sjoin_nearest properly.
    """
    print("  - Placing facilities using Hierarchical Method with Elbow-Method tuning...")

    clustered_points = gdf_clustered[gdf_clustered['cluster_id'] != -1].copy()
    if len(clustered_points) == 0: return None

    all_new_facilities = []

    for macro_cluster_id, group in tqdm(clustered_points.groupby('cluster_id'), total=len(clustered_points['cluster_id'].unique()), desc="  Processing Macro-Clusters"):
        group = group.copy()
        coords = np.array(list(group.geometry.apply(lambda p: (p.x, p.y))))

        if len(group) < 20:
             k = math.ceil(len(group) / config["HABS_PER_FACILITY_THRESHOLD"])
        else:
            print(f"\n    - Finding optimal k for macro-cluster {macro_cluster_id} (size: {len(group)})...")
            k = find_optimal_k_elbow(coords, max_k=15)
            print(f"    - Optimal k found: {k}")

        if k == 1:
            group['sub_cluster_id'] = 0
        else:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto').fit(coords)
            group['sub_cluster_id'] = kmeans.labels_

        sub_cluster_geoms = group.dissolve(by='sub_cluster_id', aggfunc={'HAB_ID': 'count'})
        sub_cluster_centroids = gpd.GeoDataFrame(geometry=sub_cluster_geoms.centroid)
        sub_cluster_centroids['hab_count'] = sub_cluster_geoms['HAB_ID'].values

        joined = gpd.sjoin_nearest(sub_cluster_centroids, nodes_gdf, how='left')
        snapped_facilities = joined.groupby(joined.index).first()


        new_facilities_for_group = gpd.GeoDataFrame({
            'nodeID': snapped_facilities['nodeID'],
            'geometry': snapped_facilities['geometry'],
            'hab_count': snapped_facilities['hab_count'],
            'parent_cluster_id': macro_cluster_id
        })


        all_new_facilities.append(new_facilities_for_group)

    if not all_new_facilities:
        raise ValueError("Hierarchical placement failed to generate any facilities.")

    gdf_facilities = pd.concat(all_new_facilities, ignore_index=True)
    gdf_facilities.drop_duplicates(subset=['nodeID'], inplace=True)
    gdf_facilities['sub_cluster_id'] = range(len(gdf_facilities))

    print(f"\n  - Generated {len(gdf_facilities)} optimal facility locations using tuned hierarchical method.")
    return gdf_facilities

def create_graph_from_gdf_optimized(gdf):
    """Creates a networkx.Graph from a GeoDataFrame of LineStrings."""
    print("  - Building network graph from GeoDataFrame...")
    gdf = gdf[gdf.geometry.is_valid & ~gdf.geometry.is_empty].copy()
    gdf.reset_index(drop=True, inplace=True)

    endpoints = gdf.geometry.boundary.explode(index_parts=False)
    unique_endpoints = endpoints.union_all()

    nodes_gdf = gpd.GeoDataFrame(geometry=list(unique_endpoints.geoms), crs=gdf.crs)
    nodes_gdf['nodeID'] = range(len(nodes_gdf))
    net = nx.Graph()
    for _, row in nodes_gdf.iterrows():
        net.add_node(row['nodeID'], geometry=row['geometry'])
    starts = gpd.GeoDataFrame(geometry=gdf.geometry.apply(lambda g: Point(g.coords[0])), crs=gdf.crs)
    ends = gpd.GeoDataFrame(geometry=gdf.geometry.apply(lambda g: Point(g.coords[-1])), crs=gdf.crs)
    start_nodes = gpd.sjoin_nearest(starts, nodes_gdf, how='left')['nodeID']
    end_nodes = gpd.sjoin_nearest(ends, nodes_gdf, how='left')['nodeID']
    gdf['u'] = start_nodes
    gdf['v'] = end_nodes
    gdf['length'] = gdf.geometry.length
    edge_data = gdf[['u', 'v', 'length']].dropna().values.tolist()
    net.add_weighted_edges_from(edge_data, weight='length')
    return net, nodes_gdf

def check_for_nan(series, column_name, error_message):
    """Checks a Pandas Series for any NaN values and stops execution if found."""
    if series.isna().any():
        num_nan = series.isna().sum()
        raise ValueError(f"[CRITICAL ERROR] Found {num_nan} NaN values in '{column_name}'. Details: {error_message}")
    print(f"  - Check successful: No NaN values found in '{column_name}'.")

def optimize_datatypes(gdf):
    """Downcasts integer and float columns in a GeoDataFrame to save memory."""
    for col in gdf.select_dtypes(include=['int64']).columns:
        gdf[col] = pd.to_numeric(gdf[col], downcast='integer')
    for col in gdf.select_dtypes(include=['float64']).columns:
        gdf[col] = pd.to_numeric(gdf[col], downcast='float')
    return gdf
from sklearn.cluster import KMeans
from kneed import KneeLocator
import warnings

def find_optimal_k_elbow(group_coords, max_k=15, min_k=1):
    """
    Finds the optimal number of clusters (k) for a set of coordinates using the elbow method.
    """
    if len(group_coords) <= max_k:
        max_k = len(group_coords) - 1
    if max_k < min_k:
        return min_k

    wcss = []
    k_range = range(min_k, max_k + 1)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        for k in k_range:
            kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto').fit(group_coords)
            wcss.append(kmeans.inertia_)

    if len(wcss) < 2:
        return min_k #

    try:
        kn = KneeLocator(list(k_range), wcss, curve='convex', direction='decreasing')
        optimal_k = kn.elbow
        if optimal_k is None:

            return min_k
        return int(optimal_k)
    except Exception:

        return min_k


def evaluate_solution_with_silhouette(gdf_habitations_final, sample_size):
    """
    Calculates the Silhouette Score using fast and memory-efficient EUCLIDEAN distance.
    This provides a geometric quality score for the final service areas.
    """
    print("\n--- Evaluating Final Solution Quality (Silhouette Score - Euclidean Method) ---")

    df = gdf_habitations_final.dropna(subset=['assigned_facility_node', 'geometry']).copy()

    if len(df) < 10:
        print("  - Not enough data points to calculate a meaningful Silhouette Score.")
        return

    if len(df) > sample_size:
        print(f"  - Dataset is large ({len(df)} points). Using a random sample of {sample_size} for evaluation.")
        df_sample = df.sample(n=sample_size, random_state=42)
    else:
        df_sample = df


    coords = np.array(list(df_sample.geometry.apply(lambda p: (p.x, p.y))))

    labels = df_sample['assigned_facility_node'].astype('category').cat.codes

    if len(np.unique(labels)) < 2:
        print(f"  - Only found {len(np.unique(labels))} unique cluster(s) in the sample. Cannot calculate Silhouette Score.")
        return

    print("  - Calculating Silhouette Score using Euclidean distances...")
    score = silhouette_score(coords, labels, metric='euclidean')

    print("\n--- Silhouette Score Result ---")
    print(f"The Euclidean-based Silhouette Score is: {score:.4f}")
    print("  - Score close to +1: Excellent. Geometric service areas are dense and well-separated.")
    print("  - Score close to 0:  Mediocre. Geometric service areas overlap significantly.")
    print("  - Score close to -1: Poor. Habitations are geometrically closer to other service areas.")
    print("="*80)

### 1. load_and_prepare_data

#### Description:
This function is responsible for loading the primary habitations shapefile, standardizing the column names, and filtering the dataset to isolate only the features classified as "Underserved."

#### Role:
As the initial data ingestion step, this function prepares the foundational dataset for the entire analysis. By filtering for "Underserved" habitations, it ensures that all subsequent processing, such as clustering and network analysis, is focused exclusively on the target population of interest, thereby reducing computational overhead and refining the analysis scope.

### Detailed Process:
#### Load Data:
gpd.read_file(filepath) reads the shapefile into a GeoPandas GeoDataFrame.
#### Standardize Column:
It checks if the service_level_field (e.g., "service_le") exists. If so, it renames it to a generic service_level for consistent use throughout the script. If not, it raises a KeyError to halt execution, preventing downstream errors.
#### Filter for Target:
It selects only the rows where the service_level column has the value 'Underserved'. The .copy() method is used to prevent a SettingWithCopyWarning in Pandas.
#### Handle Empty Data:
 If no "Underserved" habitations are found, the function returns None to allow the main script to handle this case gracefully.
#### Return Filtered Data:
The function returns the filtered GeoDataFrame containing only the habitations relevant to the analysis.
### 2. perform_dbscan_clustering
#### Description:
This function applies the Density-Based Spatial Clustering of Applications with Noise (DBSCAN) algorithm to the geographic coordinates of the underserved habitations.

#### Role:

This is the core analytical step for identifying areas of need. The function groups geographically proximate habitations into distinct clusters. This process transforms a scattered set of individual points into meaningful, aggregated zones, which can then be targeted for new service infrastructure. It also identifies isolated habitations as "outliers."

### Detailed Process:
#### Extract Coordinates:
The geometry column is used to extract the (x, y) coordinates for each point, which are then converted into a NumPy array, the required input format for Scikit-learn's DBSCAN.
#### Initialize & Run DBSCAN:
A DBSCAN object is instantiated using the eps (search radius) and min_samples (density threshold) parameters from the configuration. The algorithm='ball_tree' is an optimization well-suited for spatial coordinate data. The .fit(coords) method executes the clustering.
#### Assign Cluster IDs:
The results of the clustering are stored in the db.labels_ attribute. A new column, cluster_id, is added to the GeoDataFrame to store these labels for each habitation. Outliers are assigned a label of -1.
Report Results: The function calculates and prints the total number of clusters found (excluding the outlier group) and the total count of outliers, providing a summary of the clustering outcome.
Return Clustered Data: It returns the GeoDataFrame, now augmented with the cluster_id for each feature.
### 3. calculate_cluster_centroids
#### Description:
This function processes the clustered habitation data to calculate the geometric centroid for each valid cluster (i.e., not the outliers).

#### Role:
The primary role of this function is to determine a single, mathematically optimal location for a potential new facility that would serve all habitations within a given cluster. These centroids represent "ideal facility locations" based on the spatial distribution of demand.

Detailed Process:
Isolate Clusters: It first filters out the outliers by selecting only points where cluster_id is not -1.
#### Dissolve Geometries:
The dissolve(by='cluster_id') operation is a powerful GeoPandas tool that merges all geometries belonging to the same cluster into a single MultiPoint geometry. The aggfunc argument simultaneously counts the number of original habitations that form each cluster.
#### Calculate Centroids:
The .centroid property is then used on these dissolved MultiPoint geometries to calculate the geometric center of each cluster.
#### Create Facilities GeoDataFrame:
A new GeoDataFrame (gdf_facilities) is created where each feature's geometry is one of the calculated centroids.
#### Add Attributes:
The hab_count (number of habitations in the cluster) is added as an attribute to the new facilities GeoDataFrame.
Return Facility Locations: The function returns the GeoDataFrame of ideal facility locations.
### 4. add_points_to_network
#### Description:
This function performs "graph surgery" by integrating a set of points (e.g., habitations or facility centroids) into an existing road network GeoDataFrame. It achieves this by finding the nearest road segment for each point and splitting that segment at the projected location of the point.

#### Role:
This is a critical pre-processing step for network analysis. A standard road network only has nodes at intersections. For accurate routing from a facility to a habitation, both the facility and the habitation must exist as nodes on the network. This function modifies the network topology to ensure these points are seamlessly integrated, enabling valid shortest-path calculations.

#### Detailed Process:
Find Nearest Edges: gpd.sjoin_nearest efficiently identifies the closest road segment (edge) for every input point.
Group Points by Edge: The results are grouped by the road segment's index. This creates a dictionary (edges_to_split) where each key is a road segment that needs to be split, and the value is a list of all points that must be snapped to it.
### Split Edges: The code iterates through each road segment identified for splitting.
For each point to be snapped, nearest_points finds the exact coordinate on the line closest to the point.
A MultiPoint geometry is created from all these "snapped" coordinates.
shapely.ops.split uses this MultiPoint geometry to break the original line segment into multiple smaller segments.
Reconstruct Network: The final network is constructed by combining the newly created split segments with all the original road segments that were not modified. pd.concat is used to merge these two sets into a final, complete GeoDataFrame.
### 5. create_graph_from_gdf_optimized
#### Description:
This function converts a GeoDataFrame of LineStrings (the road network) into a networkx.Graph object, which is a mathematical representation of a network.

#### Role:
To translate the visual, geometric road data into an abstract graph structure (nodes and edges with weights) that network analysis algorithms can operate on. The resulting graph is the final data structure used for computing shortest paths, service areas, and other connectivity metrics.

#### Detailed Process:
Extract Unique Nodes: It first identifies all unique start and end points of every line segment in the GeoDataFrame. This is done efficiently using boundary.explode followed by unary_union. These unique points represent the network's nodes (intersections and endpoints).
#### Create Graph and Add Nodes:
An empty networkx.Graph is initialized. The unique nodes identified in the previous step are added to the graph, with their shapely.Point geometry stored as a node attribute.
#### Map Edges to Nodes:
For each line segment (edge), the function identifies its start and end coordinates. It then uses sjoin_nearest to find the corresponding nodeID from the previously created nodes GeoDataFrame. This robustly assigns a source (u) and target (v) node ID to each edge.
#### Add Weighted Edges:
Finally, the function calculates the length of each road segment and adds it as a length attribute. It then adds all edges to the graph using net.add_weighted_edges_from, where the edge's weight is its length. This weight is crucial for shortest path calculations.

## Main Analysis Function:  Breakdown

In [ ]:
def run_facility_proposal_analysis(config, eps, min_samples):
    start_time = time.time()
    print(f"--- Starting Facility Proposal Analysis (eps={eps}, min_samples={min_samples}) ---")


    print("\n- Stage 1: Loading and Preparing Data")
    gdf_underserved = load_and_prepare_data(config["INPUT_SHAPEFILE"], config["SERVICE_LEVEL_FIELD"])
    if gdf_underserved is None:
        print("No underserved habitations found. Stopping analysis.")
        return None
    gdf_underserved = gdf_underserved.to_crs(config["TARGET_CRS"])
    gdf_underserved = optimize_datatypes(gdf_underserved)


    print("\n- Stage 2: Performing Macro-Clustering (DBSCAN)")
    gdf_clustered, n_clusters, n_outliers = perform_dbscan_clustering(gdf_underserved, eps, min_samples)
    del gdf_underserved; gc.collect()

    print("\n- Stage 3: Prepare Network and Map Habitations to Nodes")
    roads_gdf = gpd.read_file(config["ROAD_NETWORK_SHAPEFILE"]).to_crs(config["TARGET_CRS"])
    prelim_graph, nodes_gdf_prelim = create_graph_from_gdf_optimized(roads_gdf)
    hab_nodes_initial = gpd.sjoin_nearest(gdf_clustered, nodes_gdf_prelim, how='left').groupby(gdf_clustered.index).first()


    gdf_clustered['nodeID'] = hab_nodes_initial['nodeID']
    check_for_nan(gdf_clustered['nodeID'], 'habitation nodeID', "Initial mapping of habitations to nodes failed.")

    print("\n- Stage 4: Place Facilities Using Hierarchical Method")
    gdf_facilities_ideal = hierarchical_facility_placement(gdf_clustered, config["HABITATION_ID_FIELD"], prelim_graph, nodes_gdf_prelim, config)

    print("\n- Stage 5: Unifying the Network by Bridging Disconnected Components")
    components = sorted(list(nx.connected_components(prelim_graph)), key=len, reverse=True)
    if 'initial_componentID' not in gdf_clustered.columns:
         gdf_clustered['initial_componentID'] = gdf_clustered['nodeID'].map({node: i for i, comp in enumerate(components) for node in comp})
    main_component_id = 0
    components_with_habs = gdf_clustered[(gdf_clustered['initial_componentID'] != main_component_id) & (gdf_clustered['cluster_id'] != -1)]['initial_componentID'].unique()
    augmented_network_gdf = roads_gdf.copy()
    if len(components_with_habs) > 0:
        print(f"  - Found {len(components_with_habs)} inhabited, disconnected components to bridge.")
        main_component_nodes = components[0]
        main_network_geom = nodes_gdf_prelim[nodes_gdf_prelim['nodeID'].isin(main_component_nodes)].geometry.union_all()
        bridges = []
        for cid in tqdm(components_with_habs, desc="    Building Bridges"):
            if pd.isna(cid): continue
            component_nodes = components[int(cid)]
            component_geom = nodes_gdf_prelim[nodes_gdf_prelim['nodeID'].isin(component_nodes)].geometry.union_all()
            bridge = LineString(nearest_points(component_geom, main_network_geom))
            bridges.append(bridge)
        gdf_bridges = gpd.GeoDataFrame(geometry=bridges, crs=config["TARGET_CRS"])
        augmented_network_gdf = pd.concat([roads_gdf, gdf_bridges], ignore_index=True)
        print(f"  - Added {len(gdf_bridges)} bridges to unify the network.")
    del prelim_graph, nodes_gdf_prelim, roads_gdf; gc.collect()


    print("\n- Stage 6: Final Pathfinding on the Unified Network")
    final_graph, nodes_gdf_final = create_graph_from_gdf_optimized(augmented_network_gdf)
    del augmented_network_gdf; gc.collect()

    print("  - Mapping all points to the final unified graph...")


    if 'nodeID' in gdf_clustered.columns: gdf_clustered = gdf_clustered.drop('nodeID', axis=1)
    hab_nodes_final = gpd.sjoin_nearest(gdf_clustered, nodes_gdf_final, how='left').groupby(gdf_clustered.index).first()

    gdf_clustered['final_nodeID'] = hab_nodes_final['nodeID']
    check_for_nan(gdf_clustered['final_nodeID'], 'habitation final_nodeID', "Final mapping of habitations failed.")
    gdf_facilities_ideal.crs = config["TARGET_CRS"]
    fac_nodes_final = gpd.sjoin_nearest(gdf_facilities_ideal, nodes_gdf_final, how='left')

    gdf_facilities_ideal['final_nodeID'] = fac_nodes_final.loc[gdf_facilities_ideal.index, 'nodeID_right']
    check_for_nan(gdf_facilities_ideal['final_nodeID'], 'facility final_nodeID', "New facilities could not be mapped.")
    new_facility_nodes = gdf_facilities_ideal['final_nodeID'].dropna().astype(int).tolist()

    gdf_facilities_original = gpd.read_file(config["ORIGINAL_FACILITIES_SHAPEFILE"]).to_crs(config["TARGET_CRS"])
    orig_fac_nodes = gpd.sjoin_nearest(gdf_facilities_original, nodes_gdf_final, how='left').groupby(gdf_facilities_original.index).first()

    existing_facility_nodes = orig_fac_nodes['nodeID'].dropna().astype(int).tolist()

    del nodes_gdf_final, hab_nodes_final, fac_nodes_final, orig_fac_nodes; gc.collect()


    all_facility_nodes = list(set(new_facility_nodes + existing_facility_nodes))
    print(f"  - Analyzing paths to {len(all_facility_nodes)} total facilities ({len(existing_facility_nodes)} existing, {len(new_facility_nodes)} new).")

    print("  - Calculating distances and facility assignments...")
    gdf_clustered['net_dist_m'] = np.nan
    gdf_clustered['assigned_facility_node'] = np.nan

    distances, paths = nx.multi_source_dijkstra(final_graph, sources=all_facility_nodes, weight='length')

    clustered_habs_nodes_series = gdf_clustered.loc[gdf_clustered['cluster_id'] != -1, 'final_nodeID']

    final_distances = clustered_habs_nodes_series.map(distances)
    check_for_nan(final_distances, 'final_distances', "Pathfinding failed for some habitations.")
    gdf_clustered.loc[final_distances.index, 'net_dist_m'] = final_distances

    assigned_nodes = clustered_habs_nodes_series.map(lambda node: paths.get(node, [np.nan])[0])
    gdf_clustered.loc[assigned_nodes.index, 'assigned_facility_node'] = assigned_nodes

    if 'path_len_m' in gdf_clustered.columns:
        gdf_clustered.loc[:, 'net_reduc_m'] = gdf_clustered['path_len_m'] - gdf_clustered['net_dist_m']

    total_time = (time.time() - start_time) / 60
    print(f"\n--- Analysis Complete ---")
    print(f"Total execution time: {total_time:.2f} minutes.")

    return gdf_clustered, gdf_facilities_ideal, final_graph, n_clusters, n_outliers

### Initial Setup
#### Description:
The function begins by recording the start time for performance tracking and printing an introductory message that includes the specific eps and min_samples hyperparameters being used for the current run.

#### Role:
This provides clear logging for traceability and allows for easy comparison of execution times when experimenting with different clustering parameters.

### Stage 1: Loading and Preparing Data
#### Description:
This stage handles the initial data ingestion by calling the load_and_prepare_data function. It then reprojects the resulting GeoDataFrame to the project's target Coordinate Reference System (CRS).

#### Role:
It ensures that the analysis begins with a clean, filtered dataset containing only the "Underserved" habitations of interest. Reprojecting the data to a projected CRS (e.g., one using meters) is a critical step that validates the integrity of all subsequent distance-based calculations.

### Stage 2: Performing DBSCAN Clustering
#### Description:
This stage executes the DBSCAN clustering algorithm on the prepared underserved habitation data by calling perform_dbscan_clustering.

#### Role:
This is the primary discovery phase of the analysis. It groups the individual underserved habitations into geographically dense clusters, effectively identifying "hotspots" of need. The function also handles the edge case where no clusters are found, allowing the analysis to terminate gracefully.

### Stage 3: Calculating Cluster Centroids
#### Description:
This stage calculates the geometric center, or centroid, of each cluster identified in the previous stage by calling calculate_cluster_centroids.

#### Role:
It generates a set of "ideal" locations for new facilities. Each centroid represents a theoretically optimal point that minimizes the straight-line distance to all habitations within its respective cluster. These points serve as the initial proposals before being validated against the real-world road network.

### Stage 5: Load and Prepare Road Network
#### Description:
This stage marks the beginning of the "Real World Validation" phase. It loads the raw road network shapefile, reprojects it to the target CRS, and filters it to include only relevant road types (e.g., primary, secondary, residential), excluding minor paths or tracks.

#### Role:
This step establishes the real-world constraints for the analysis. By filtering the network, it improves both performance and the realism of the results, ensuring that routing calculations are based on viable transportation infrastructure.

### Stage 5.5: Integrating All Points into the Road Network
#### Description:
This critical stage performs "graph surgery." It combines the locations of all underserved habitations and the newly proposed facility centroids into a single set of points. It then calls add_points_to_network to integrate these points into the road network by splitting existing road segments.

#### Role:
This is arguably the most important step for ensuring network analysis accuracy. A point must exist as a node in the network graph to be a valid origin or destination for routing. This function modifies the network topology so that every habitation and facility is a connectable point, enabling precise shortest-path calculations.

### Stage 6: Calculating Network Distances (Optimized)
This stage is the computational core of the validation process. It calculates the actual travel distance along the road network from each proposed facility to the habitations it is intended to serve.

#### Detailed Process:
Graph Creation: Calls create_graph_from_gdf_optimized to convert the integrated road network GeoDataFrame into a networkx.Graph object, where intersections are nodes and road segments are edges weighted by their length.

### Network Component Analysis (Optimization):

#### Description:
It identifies all disconnected sub-networks (components) within the graph (e.g., an island's road network separate from the mainland).
####  Role:
This is a major optimization. It prevents the algorithm from wasting time trying to calculate a path between two points that are physically disconnected. A node_to_component dictionary is created for fast lookups.
Mapping Points to Graph Nodes:

#### Description:
It uses gpd.sjoin_nearest to find the closest graph node for every habitation and every facility.
Role: This step bridges the gap between the geographic coordinates of the points and their corresponding abstract representations within the graph structure.
Component ID Assignment: Using the node_to_component map, it assigns a componentID to each habitation and facility. This allows for rapid checking of whether a facility and a habitation are on the same connected network.

### Path Calculation via Intelligent Pruning:

### Description:
 The code iterates through each cluster. For a given cluster, it identifies the proposed facility and its associated network component. It then filters for only the habitations belonging to that same component.
Role: This is the payoff of the component analysis. By creating a temporary subgraph containing only the relevant component, the pathfinding algorithm (Dijkstra's) runs on a much smaller, simpler network, drastically improving performance.
Algorithm: nx.single_source_dijkstra_path_length is used. This is highly efficient as it calculates the shortest path from the facility node to all other nodes in the component in a single pass. The results are then mapped back to the relevant habitation nodes.
Result Aggregation: The calculated network distances (net_dist_m) from all clusters are concatenated and merged back into the main gdf_clustered GeoDataFrame.

### Conclusion and Return
#### Description:
The function calculates the total execution time and prints a completion summary. It then returns a tuple containing all the key artifacts generated during the analysis.

#### Role:
It provides a comprehensive set of outputs for further analysis, visualization, or reporting, including:

#### gdf_clustered:
The original habitations with added cluster IDs and calculated network distances.
#### gdf_facilities_ideal:
The proposed facility locations with their attributes.
#### final_network_gdf:
The modified road network GeoDataFrame with all points integrated.
#### n_clusters, n_outliers:
Summary statistics from the clustering step.

## Save Function : Breakdown

In [ ]:
def save_outputs(config, gdf_clustered, gdf_facilities, final_graph, eps, min_samples):
    """Saves all output files to the specified directory."""
    print("\n" + "="*80)
    print("--- Stage 7: Saving All Output Files ---")
    outlier_count = len(gdf_clustered[gdf_clustered['cluster_id'] == -1])
    if outlier_count > 0:
        print(f"  - Removing {outlier_count} outliers from the final habitations shapefile.")
    gdf_clustered_save = gdf_clustered[gdf_clustered['cluster_id'] != -1].copy()
    gdf_facilities_save = gdf_facilities.copy()

    gdf_clustered_save.columns = [str(col)[:10] for col in gdf_clustered_save.columns]
    gdf_facilities_save.columns = [str(col)[:10] for col in gdf_facilities_save.columns]

    habs_output_dir = os.path.join(config["OUTPUT_DIR"], config["HABITATIONS_FOLDER"])
    os.makedirs(habs_output_dir, exist_ok=True)
    habs_filename = f"habs_proposals_eps{eps}_mins{min_samples}_validated.shp"
    habs_path = os.path.join(habs_output_dir, habs_filename)
    gdf_clustered_save.to_file(habs_path, driver='ESRI Shapefile')
    print(f"  - Saved updated habitations to: {habs_path}")

    facs_output_dir = os.path.join(config["OUTPUT_DIR"], config["FACILITIES_FOLDER"])
    os.makedirs(facs_output_dir, exist_ok=True)
    facs_filename = f"facilities_eps{eps}_mins{min_samples}_final.shp"
    facs_path = os.path.join(facs_output_dir, facs_filename)
    gdf_facilities_save.to_file(facs_path, driver='ESRI Shapefile')
    print(f"  - Saved proposed facilities to: {facs_path}")

    print("  - Rebuilding network GeoDataFrame from graph for saving...")
    net_output_dir = os.path.join(config["OUTPUT_DIR"], config["NETWORK_FOLDER"])
    os.makedirs(net_output_dir, exist_ok=True)
    final_network_path = os.path.join(net_output_dir, config["HYBRID_NETWORK_FILENAME"])
    edges = list(final_graph.edges())
    edge_geometries = [LineString([final_graph.nodes[u]['geometry'], final_graph.nodes[v]['geometry']]) for u, v in tqdm(edges, desc="    Converting Edges")]
    network_to_save = gpd.GeoDataFrame(geometry=edge_geometries, crs=config["TARGET_CRS"])
    network_to_save.to_file(final_network_path, driver='ESRI Shapefile')
    print(f"  - Saved final analysis network to: {final_network_path}")
    print("="*80)


#### Description:
This function is responsible for the persistent storage of all key analytical outputs generated by the main pipeline. It systematically saves the clustered habitations GeoDataFrame, the proposed facilities GeoDataFrame, and the final integrated network GeoDataFrame to disk as ESRI Shapefiles.

#### Role:
As the final stage of the pipeline, this function is critical for materializing the in-memory results into tangible, reusable files. This enables downstream use cases such as visualization in GIS software (e.g., QGIS, ArcGIS), inclusion in reports, and long-term archival. The function's logic also ensures that outputs are organized logically and named descriptively, which is essential for reproducibility and comparing the results from different analysis runs.

### Detailed Process
#### Initiate Saving Process:

The function begins by printing a formatted header to the console, clearly indicating that the file-saving stage has commenced. This improves the readability of the script's log output.
#### Create Defensive Copies:

gdf_clustered_save = gdf_clustered.copy(): The function first creates copies of the incoming GeoDataFrames.
Reasoning: This is a robust programming practice. The function is about to modify the data (specifically, the column names). By operating on a copy, it ensures that the original GeoDataFrame objects in the main script's memory remain unaltered, preventing any unintended side effects if they are needed for subsequent operations.
#### Truncate Column Names:

gdf_..._save.columns = [str(col)[:10] for col in ..._save.columns]: This line iterates through all column names in the copied GeoDataFrames and truncates each one to a maximum of 10 characters.
Reasoning: This is a mandatory step to ensure compatibility with the ESRI Shapefile format. The shapefile specification has a strict limitation that field (column) names cannot exceed 10 characters. Without this step, the to_file operation would raise an error for any column with a longer name (e.g., 'cluster_id', 'service_level').
#### Save Habitation Results:

#### Path Construction:
os.path.join is used to construct a platform-independent file path by combining the main OUTPUT_DIR, the HABITATIONS_FOLDER from the configuration, and a dynamically generated filename.
#### Directory Creation:
os.makedirs(..., exist_ok=True) ensures that the target output directory exists; if not, it creates it. The exist_ok=True argument prevents an error if the directory already exists from a previous run.
#### Dynamic Filenaming:
The filename is generated using an f-string: f"habs_proposals_eps{eps}_mins{min_samples}_validated.shp". This is a key feature for traceability, as it embeds the specific eps and min_samples hyperparameters used for that analysis directly into the filename.
#### Write to File:
gdf_clustered_save.to_file(...) executes the write operation, saving the data as a shapefile.
#### Save Facility Results:

This block follows the exact same logic as the habitation-saving step but is applied to the gdf_facilities_save GeoDataFrame. It creates a descriptively named shapefile for the proposed facility locations within the appropriate sub-directory.
#### Save Final Network:

This block saves the modified final_network_gdf, which includes all the integrated points.
The filename for the network is taken directly from the config dictionary (HYBRID_NETWORK_FILENAME). This is because the integrated network is often a foundational output that might be consistent across multiple runs with different clustering parameters.
#### Conclude Saving Process:

Finally, the function prints a closing separator to the console, signifying that all save operations are complete.

## **Execution Block : Beakdown**

In [ ]:
if __name__ == '__main__':
    try:
        results = run_facility_proposal_analysis(
            config=_CONFIG,
            eps=_CONFIG["DBSCAN_EPS_METERS"],
            min_samples=_CONFIG["DBSCAN_MIN_SAMPLES"]
        )
        if results:
            gdf_clustered, gdf_facilities_ideal, final_graph, n_clusters, n_outliers = results

            if 'net_reduc_m' in gdf_clustered.columns and not gdf_clustered.dropna(subset=['net_reduc_m', 'cluster_id']).empty:
                gdf_results = gdf_clustered.dropna(subset=['net_reduc_m'])
                gdf_improved = gdf_results[gdf_results['net_reduc_m'] > 0].copy()
                if not gdf_improved.empty:
                    num_total_calculated = len(gdf_results[gdf_results['cluster_id']!=-1])
                    num_improved = len(gdf_improved)
                    avg_net_reduc = gdf_improved['net_reduc_m'].mean()
                    print("\n--- Final Results Summary ---")
                    print(f"Total clustered habitations with a calculated new path: {num_total_calculated}")
                    print(f"Habitations with a POSITIVE distance reduction: {num_improved} ({num_improved/num_total_calculated:.2%})")
                    print(f"\n[SUCCESS] For the habitations that improved, the average network reduction is ({avg_net_reduc:.2f}m).")

                    evaluate_solution_with_silhouette(
                     gdf_habitations_final=gdf_clustered,
                     sample_size=_CONFIG["SILHOUETTE_SAMPLE_SIZE"]
                 )

                    save_outputs(
                        config=_CONFIG,
                        gdf_clustered=gdf_clustered,
                        gdf_facilities=gdf_facilities_ideal,
                        final_graph=final_graph,
                        eps=_CONFIG["DBSCAN_EPS_METERS"],
                        min_samples=_CONFIG["DBSCAN_MIN_SAMPLES"]
                    )
                else:
                    print("\n[WARNING] Analysis completed, but no habitations showed a positive distance reduction.")
    except Exception as e:
        print(f"\n[CRITICAL ERROR] The analysis failed: {e}")
        import traceback
        traceback.print_exc()

### 1. The if __name__ == '__main__': Guard
#### Description:
This is a standard Python construct. The special variable __name__ is automatically set to the string '__main__' when the script is executed directly. If the script is imported as a module into another script, __name__ will be set to the module's name instead.

#### Role:
This "guard" ensures that the code within this block—the actual execution of the analysis—only runs when the script is the main program being executed. This is a fundamental best practice that allows the script's functions (like load_and_prepare_data, save_outputs, etc.) to be safely imported and reused in other programs without automatically triggering a full analysis run.

### 2. The try...except Block: Robust Error Handling
#### Description:
The entire workflow is wrapped in a try...except block. The main logic is placed inside the try clause, while the except clause defines how to respond if any part of that logic fails and raises an exception (an error).

#### Role:
This structure makes the script robust and fault-tolerant. Geospatial analysis can be prone to errors (e.g., missing files, invalid data, CRS mismatches, disconnected networks). Instead of crashing and providing a cryptic error message, this block "catches" any critical failure, prints a formatted and user-friendly error report, and exits gracefully.

### 3. Executing the Analysis Pipeline
#### Description:
Inside the try block, the main function run_facility_proposal_analysis is called. It is passed the global _CONFIG dictionary and the specific DBSCAN parameters extracted from that configuration. The complete set of results is captured in the results variable.

#### Role:
This is the trigger for the entire end-to-end analysis. It initiates the sequence of data loading, clustering, network integration, and distance calculations. The results variable will either contain a tuple of the generated data artifacts or the value None if the analysis terminated early (e.g., no underserved habitations were found).

#### 4. Results Validation and Conditional Saving
This section acts as a critical quality control gate. It ensures that the outputs are not saved unless they are complete and logically sound.

### Step 1: Check for Valid Results (if results:)
####Description:
It first checks if the results variable is not None.
####Role:
 This handles cases where the analysis function returned early. If results is None, the script does nothing further, preventing an error from trying to access data that doesn't exist.
### Step 2: Unpack Results
### Description:
 If results are present, the tuple is unpacked into its constituent parts (gdf_clustered, gdf_facilities_ideal, etc.) for easier access.
### Step 3: Perform Final Sanity Checks
### Description:
 A series of nested if/else statements meticulously checks the validity of the final output.
#### if 'net_reduc_m' in gdf_clustered.columns::
It first verifies that the key output column, net_reduc_m (presumably "network reduction in meters"), was successfully created. If not, it prints a warning and stops.
#### if not avg_net_reduc_series.empty::
After filtering out any rows where a path could not be calculated (.dropna()), it checks if there are any valid path distances left to analyze. If not, it means no valid network paths could be established.
#### if pd.notna(avg_net_reduc) and avg_net_reduc > 0::
This is the ultimate business logic check. It calculates the average network distance reduction and verifies that the value is positive. A zero or negative value would imply the proposed facilities offer no travel distance benefit, indicating a potential logical error or an invalid outcome.
### Step 4: Conditional Save (save_outputs(...))
#### Description:
Only if all the preceding checks pass—meaning the analysis ran completely and produced a positive, logical result—is the save_outputs function called.
### Role:
 This ensures that only high-quality, valid results are saved to disk, preventing the clutter of incomplete or erroneous files from failed or nonsensical analysis runs.
### 5. Exception Handling (except Exception as e:)
#### Description:
If any unhandled error occurs anywhere within the try block, program control immediately jumps here.

#### Role:
This is the "fail-safe" mechanism.

It prints a clear [CRITICAL ERROR] message to the console, informing the user that the process failed catastrophically.
It includes the specific exception message ({e}) to give an initial clue as to what went wrong.
Crucially, traceback.print_exc() is called. This utility prints the full "stack trace"—a detailed report showing the exact file, function, and line number where the error occurred. This information is invaluable for developers to debug and fix the root cause of the failure.

# **Summary of Analysis Run**

 The script processed 58,067 underserved habitations and grouped them into 146 distinct clusters based on geographic proximity.

### Here is a breakdown of the key results:

#### Clustering:
 Using a 5km search radius (eps=5000) and a minimum density of 10 habitations (min_samples=10), the algorithm identified 346 viable clusters needing service. 2,643 habitations were too isolated to be clustered and were classified as outliers.
####  Proposal Generation:
One ideal facility location (a geometric centroid) was generated for each of the 346 clusters.
#### Network Validation:
These proposed locations were validated against a real-world road network containing over 1 million segments. The analysis confirmed that valid travel paths could be calculated on the network.
#### Final Outcome:
The analysis was successful and produced valid results. Three new shapefiles were saved to Google Drive:
### Habitations Output: The original habitations with their new cluster IDs.
### Facilities Output: The 346 proposed facility locations.
### Network Output: The final, modified road network used for the calculations.

#**Visualizing The Outputs on a  Map**

 ## **Loading The Map In LocalTunnel(Server App)**

In [ ]:
print("--- 1q. Setting up directories and loading data ---")
LOCAL_MAP_DIR = '/content/map_files/'
os.makedirs(LOCAL_MAP_DIR, exist_ok=True)
GDRIVE_MAP_DIR = '/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis/ServiceLevelAnalysis/Map_Dashboard/'
os.makedirs(GDRIVE_MAP_DIR, exist_ok=True)
SIMPLIFICATION_TOLERANCE = 0.001

path_hab_level = '/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis_No_Jump_Lines/ServiceLevelAnalysis/maharashtra_habitation_service_levels.shp'
path_shortest_path = '/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis_No_Jump_Lines/shortest_paths.shp'
path_proposed_fas = '/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis/HierarchicalClustering/Facilities_Output/facilities_eps5000_mins10_final.shp'
path_clustered_habs = '/content/drive/MyDrive/network_analysis_output/Maharashtra_Analysis/ServiceLevelAnalysis/HierarchicalClustering/Habitations_Output/habs_proposals_eps5000_mins10_validated.shp'
path_to_original_facilities = '/content/drive/MyDrive/Rukiya project/Facilities/Facilities_cleaned.shp'

gdf_hab_level = gpd.read_file(path_hab_level)
gdf_shortest_path = gpd.read_file(path_shortest_path)
gdf_proposed_fas = gpd.read_file(path_proposed_fas)
gdf_facilities_original = gpd.read_file(path_to_original_facilities)
gdf_clustered_habs = gpd.read_file(path_clustered_habs)



In [ ]:
target_crs_display = 'EPSG:4326'
for gdf in [gdf_hab_level, gdf_shortest_path, gdf_proposed_fas, gdf_facilities_original, gdf_clustered_habs]:
    if gdf.crs != target_crs_display:
        gdf.to_crs(target_crs_display, inplace=True)


reduction_data = gdf_clustered_habs[['HAB_ID', 'net_dist_m', 'net_reduc_']]
gdf_hab_level = gdf_hab_level.merge(reduction_data, on='HAB_ID', how='left')

In [ ]:
print("--- 1b. Preparing data for Gradio interface ---")
NAMES_FILE_PATH = os.path.join(GDRIVE_MAP_DIR, 'habitation_names.json')
FULL_DATA_PATH = os.path.join(GDRIVE_MAP_DIR, 'full_habitation_data.json')
gdf_hab_level['Habitation_Name'] = "Habitation_" + gdf_hab_level['HAB_ID'].astype(str)
habitation_names = sorted(gdf_hab_level['Habitation_Name'].dropna().unique().tolist())
with open(NAMES_FILE_PATH, 'w') as f: json.dump(habitation_names, f)
full_data_for_gradio = [{"name": r['Habitation_Name'],
                         "id": str(r['HAB_ID']),
                         "level": r.get('service_le', 'U'),
                         "orig_dist": r.get('path_len_m'),
                         "near_fac_id": r.get('nearest_fa'),
                         "net_dist_m": r.get('net_dist_m'),
                         "net_reduc_m": r.get('net_reduc_')} for _, r in gdf_hab_level.iterrows()]
with open(FULL_DATA_PATH, 'w') as f: json.dump(full_data_for_gradio, f)

In [ ]:

print("--- 2. Preparing all map layers in memory ---")
served_gdf = gdf_hab_level[gdf_hab_level['service_le'] == 'Served'].copy()
underserved_gdf = gdf_hab_level[gdf_hab_level['service_le'] == 'Underserved'].copy()
gdf_shortest_path.geometry = gdf_shortest_path.geometry.simplify(SIMPLIFICATION_TOLERANCE)

cluster_col_name = next((col for col in gdf_clustered_habs.columns if 'cluster' in col.lower()), 'cluster_id')
polygons_series = gdf_clustered_habs[gdf_clustered_habs[cluster_col_name] != -1].dissolve(by=cluster_col_name).convex_hull
gdf_polygons = gpd.GeoDataFrame(geometry=polygons_series)
print("  - All layers prepared successfully.")


print("--- 3. Creating map and adding all layers using Folium ---")
m = folium.Map(location=[19.75, 75.75], zoom_start=7, tiles='CartoDB positron')

if not served_gdf.empty:
    served_cluster = MarkerCluster(name='Served Habitation').add_to(m)
    folium.GeoJson(served_gdf, marker=folium.CircleMarker(radius=4, color='green', fill_color='green', fill_opacity=0.7), tooltip=folium.GeoJsonTooltip(fields=['Habitation_Name'])).add_to(served_cluster)
if not underserved_gdf.empty:
    underserved_cluster = MarkerCluster(name='Underserved Habitation').add_to(m)
    folium.GeoJson(underserved_gdf, marker=folium.CircleMarker(radius=4, color='red', fill_color='red', fill_opacity=0.7), tooltip=folium.GeoJsonTooltip(fields=['Habitation_Name'])).add_to(underserved_cluster)
if not gdf_facilities_original.empty:
    existing_cluster = MarkerCluster(name='Existing Facility').add_to(m)
    folium.GeoJson(gdf_facilities_original, marker=folium.Marker(icon=folium.Icon(color='purple', icon='plus-square', prefix='fa'))).add_to(existing_cluster)
if not gdf_proposed_fas.empty:
    proposed_cluster = MarkerCluster(name='Proposed Facility').add_to(m)
    folium.GeoJson(gdf_proposed_fas, marker=folium.Marker(icon=folium.Icon(color='blue', icon='star', prefix='fa'))).add_to(proposed_cluster)

if not gdf_shortest_path.empty:
    folium.GeoJson(gdf_shortest_path, name='Shortest Path', style_function=lambda x: {'color': '#3182bd', 'weight': 2, 'opacity': 0.7}, show=False).add_to(m)
if not gdf_polygons.empty:
    folium.GeoJson(gdf_polygons, name='Region of Need', style_function=lambda x: {'fillColor': '#ff7800', 'color': '#ff7800', 'weight': 2, 'fillOpacity': 0.2}, show=False).add_to(m)

print("--- 4. Adding legend and JS for URL-based zoom ---")
folium.LayerControl(collapsed=False).add_to(m)
legend_html = """<div style="position:fixed; bottom:50px; left:50px; width:220px; z-index:1000; border:2px solid grey; background-color:rgba(255, 255, 255, 0.8); border-radius:5px; padding:10px; font-size:14px;"><b>Legend</b><br><i class="fa fa-circle" style="color:green"></i>&nbsp; Served Habitation<br><i class="fa fa-circle" style="color:red"></i>&nbsp; Underserved Habitation<br><i class="fa fa-plus-square" style="color:purple"></i>&nbsp; Existing Facility<br><i class="fa fa-star" style="color:blue"></i>&nbsp; Proposed Facility<br><hr style="margin:5px 0;"><span style="background-color:#3182bd; display:inline-block; height:5px; width:20px; vertical-align:middle;"></span>&nbsp; Shortest Path<br><span style="background-color:rgba(255, 120, 0, 0.5); border:1px solid #ff7800; display:inline-block; height:10px; width:20px; vertical-align:middle;"></span>&nbsp; Region of Need</div>"""
m.get_root().html.add_child(folium.Element(f'<link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.min.css">'))
m.get_root().html.add_child(folium.Element(legend_html))

habitation_index = {str(row['HAB_ID']): [row.geometry.y, row.geometry.x] for _, row in gdf_hab_level.iterrows() if pd.notna(row.geometry)}
habitation_index_json = json.dumps(habitation_index)
url_based_zoom_js = f"""
<script>
    // These variables are defined by Folium before this script runs.
    var map = {m.get_name()};
    const habitationIndex = {habitation_index_json};

    function zoomToHabitation(habId) {{
        const coords = habitationIndex[habId];
        if (!coords) {{
            console.error("FAILED: Habitation ID '" + habId + "' not found in the index.");
            return;
        }}
        console.log("SUCCESS: Found coordinates for " + habId + ". Zooming now.");

        // Use a setTimeout to ensure the map has fully initialized and rendered
        // before we try to fly to a location. This is crucial.
        setTimeout(function() {{
             map.flyTo(coords, 14);
        }}, 500); // 500ms delay
    }}

    // --- THIS IS THE MAIN EXECUTION LOGIC ---
    // This code runs as soon as the script tag is loaded by the browser.

    console.log("Map script loaded. Checking for URL parameters...");
    console.log("Current URL:", window.location.href);

    const params = new URLSearchParams(window.location.search);
    const habIdToZoom = params.get('hab_id');

    if (habIdToZoom) {{
        console.log("Found 'hab_id' parameter in URL:", habIdToZoom);
        // We found an ID, so we call the function to zoom.
        zoomToHabitation(habIdToZoom);
    }} else {{
        console.log("No 'hab_id' parameter found in URL. Displaying default map view.");
    }}
</script>
"""
m.get_root().html.add_child(folium.Element(url_based_zoom_js))
Fullscreen().add_to(m)

MAP_FILE_PATH = os.path.join(LOCAL_MAP_DIR, 'index.html')
m.save(MAP_FILE_PATH)
print(f"✅ Map with URL-based zoom is ready in {LOCAL_MAP_DIR}")
print("\n--- 8. Starting Server, Tunnel, and Capturing URL for Gradio ---")
os.makedirs(LOCAL_MAP_DIR, exist_ok=True)

with open(os.path.join(LOCAL_MAP_DIR, "placeholder.html"), "w") as f:
    f.write("Server is running.")

!npm install -g localtunnel -q

public_url = ""
tunnel_password = ""
PORT = 8000

def start_server():
    os.chdir(LOCAL_MAP_DIR)
    handler = http.server.SimpleHTTPRequestHandler
    try:
        with socketserver.TCPServer(("", PORT), handler) as httpd:
            print(f"✅ Local map server running at http://localhost:{PORT}")
            httpd.serve_forever()
    except OSError as e:
        print(f"⚠️ Server port {PORT} is already in use. Assuming server is running from a previous cell execution.")
        pass

server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(2)

try:
    tunnel_password = requests.get('https://api.ipify.org').text
except Exception:
    tunnel_password = "Could not be determined"

lt_process = subprocess.Popen(['lt', '--port', str(PORT)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

print("Waiting for localtunnel to provide a public URL...")
for line in iter(lt_process.stdout.readline, ''):
    print(f"[localtunnel] {line.strip()}")
    if "your url is:" in line:
        match = re.search(r'(https://[a-zA-Z0-9-]+.loca.lt)', line)
        if match:
            public_url = match.group(0)
            break

if public_url:
    print("\n" + "="*70)
    print(" ✅ SUCCESS! URL AND PASSWORD CAPTURED FOR AUTOMATION.")
    print(f"    URL: {public_url}")
    print(f"    Password: {tunnel_password}")
    print("\n >> You can now run the next cell to launch the Gradio Dashboard.")
    print("="*70)
else:
    print("❌ Could not get localtunnel URL. Automation will fail.")


## **Gradio Interface and Map Building**

In [ ]:
print("\n--- 5. Preparing analytics and plots for the dashboard ---")


total_habs, num_served, num_underserved = len(gdf_hab_level), len(served_gdf), len(underserved_gdf)
underserved_distances_km = underserved_gdf['path_len_m'] / 1000
path_type_counts = gdf_hab_level['path_type'].value_counts()

paths_found = path_type_counts.sum()
paths_optimal = path_type_counts.get('Nearest Found', 0)
paths_fallback = paths_found - paths_optimal

def create_service_level_plot():
    """Generates a pie chart of the service level distribution."""
    plt.style.use('seaborn-v0_8-talk'); fig, ax = plt.subplots(figsize=(7, 5))
    if num_underserved > 0 or num_served > 0:
        ax.pie([num_underserved, num_served], explode=(0.1, 0), labels=['Underserved', 'Served'], colors=['#FF6347', '#3CB371'], autopct='%1.1f%%', shadow=True, startangle=140, textprops={'fontsize': 14, 'fontweight': 'bold'})
        ax.set_title('Service Level Distribution', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout(); return fig

def create_distance_histogram():
    """Generates a histogram of distances for underserved habitations."""
    plt.style.use('seaborn-v0_8-talk'); fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(underserved_distances_km[underserved_distances_km <= 50], kde=True, ax=ax, color='salmon', bins=30)
    ax.set_title('Distance Profile for Underserved Habitations', fontsize=16, fontweight='bold'); ax.set_xlabel('Distance to Nearest Facility (km)', fontsize=14); ax.set_ylabel('Number of Habitations', fontsize=14)
    avg_dist = underserved_distances_km.mean(); ax.axvline(avg_dist, color='red', linestyle='--', linewidth=2.5)
    ax.text(avg_dist * 1.05, ax.get_ylim()[1] * 0.9, f'Avg: {avg_dist:.1f} km', color='red', fontweight='bold', fontsize=14)
    plt.tight_layout(); return fig

def create_path_type_plot():
    """Generates a bar chart showing the breakdown of pathfinding methods used."""
    plt.style.use('seaborn-v0_8-talk'); fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(x=path_type_counts.index, y=path_type_counts.values, ax=ax, palette="viridis")
    ax.set_title('Breakdown of Pathfinding Methods', fontsize=16, fontweight='bold')
    ax.set_xlabel('Path Type', fontsize=14)
    ax.set_ylabel('Number of Habitations', fontsize=14)
    ax.tick_params(axis='x', rotation=10, labelsize=12)
    plt.tight_layout(); return fig

def create_dijkstra_explanation_plot():
    """Generates a conceptual diagram of Dijkstra's algorithm."""
    plt.style.use('seaborn-v0_8-talk'); G = nx.Graph()
    edges = [('A','B',7), ('A','C',4), ('B','C',2), ('B','D',5), ('C','D',6), ('C','E',3), ('D','F',8), ('E','F',5)]
    G.add_weighted_edges_from(edges); pos = nx.spring_layout(G, seed=42)
    path = ['A', 'C', 'E', 'F']; path_edges = list(zip(path, path[1:]))
    fig, ax = plt.subplots(figsize=(8, 5)); nx.draw_networkx_nodes(G, pos, node_color='skyblue', node_size=1500, ax=ax)
    nx.draw_networkx_labels(G, pos, font_size=16, font_weight='bold', ax=ax)
    nx.draw_networkx_edges(G, pos, edge_color='gray', width=2, ax=ax)
    nx.draw_networkx_edges(G, pos, edgelist=path_edges, edge_color='red', width=4, ax=ax)
    edge_labels = nx.get_edge_attributes(G, 'weight'); nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=14, ax=ax)
    ax.set_title("Dijkstra's Algorithm: Finding the Shortest Path", fontsize=16, fontweight='bold')
    ax.text(0.05, 0.05, 'Path A→C→E→F = 4+3+5 = 12 (Shortest)', transform=ax.transAxes, color='red', fontsize=14, fontweight='bold')
    plt.tight_layout(); return fig


num_dbscan_clusters, num_dbscan_outliers, num_final_facilities, silhouette_score = 146, 2643, 381, -0.0962
def create_elbow_method_explanation_plot():
    plt.style.use('seaborn-v0_8-talk'); k=np.arange(1,11); inertia=1000/k+np.random.rand(10)*100/k; elbow_point=4
    fig, ax = plt.subplots(figsize=(8, 5)); ax.plot(k, inertia, 'bx-'); ax.set_xlabel('Number of Sub-Clusters (k)', fontsize=14); ax.set_ylabel('WCSS (Inertia)', fontsize=14)
    ax.set_title('The Elbow Method for Finding Optimal k', fontsize=16, fontweight='bold'); ax.axvline(x=elbow_point, color='r', linestyle='--', label=f'Elbow Point (Optimal k = {elbow_point})')
    ax.annotate('The "Elbow": Point of diminishing returns', xy=(elbow_point, inertia[elbow_point-1]), xytext=(elbow_point + 1, inertia[elbow_point-1] + 200), arrowprops=dict(facecolor='black', shrink=0.05), fontsize=12)
    ax.legend(fontsize=12); plt.tight_layout(); return fig

benefiting_habs_df = gdf_hab_level[gdf_hab_level['net_reduc_'].notna() & (gdf_hab_level['net_reduc_'] > 0)].copy()
benefiting_habs_df['original_dist_km'] = benefiting_habs_df['path_len_m'] / 1000; benefiting_habs_df['new_dist_km'] = benefiting_habs_df['net_dist_m'] / 1000
habs_newly_served = len(benefiting_habs_df[(benefiting_habs_df['service_le'] == 'Underserved') & (benefiting_habs_df['new_dist_km'] <= 5)])
num_habs_benefiting = len(benefiting_habs_df); avg_dist_reduction_m = benefiting_habs_df['net_reduc_'].mean()
def create_before_after_dist_plot():
    plt.style.use('seaborn-v0_8-talk'); fig, ax = plt.subplots(figsize=(10, 5))
    sns.kdeplot(data=benefiting_habs_df, x='original_dist_km', fill=True, alpha=0.5, label='Before', ax=ax, color='tomato', cut=0)
    sns.kdeplot(data=benefiting_habs_df, x='new_dist_km', fill=True, alpha=0.5, label='After', ax=ax, color='forestgreen', cut=0)
    ax.set_title('Impact of New Facilities on Travel Distance', fontsize=16, fontweight='bold'); ax.set_xlabel('Distance to Nearest Facility (km)', fontsize=14); ax.set_ylabel('Density of Habitations', fontsize=14)
    ax.legend(fontsize=12); plt.tight_layout(); return fig
def create_impact_summary_plot():
    plt.style.use('seaborn-v0_8-talk'); fig, ax = plt.subplots(figsize=(7, 5))
    metrics = ['Habitations Benefiting', 'Newly "Served"']; values = [num_habs_benefiting, habs_newly_served]
    bars = ax.bar(metrics, values, color=['#4682B4', '#3CB371']); ax.set_title('Key Proposal Outcomes', fontsize=16, fontweight='bold'); ax.set_ylabel('Number of Habitations', fontsize=14)
    ax.tick_params(axis='x', labelsize=14); ax.set_ylim(0, max(values) * 1.15)
    for bar in bars: yval = bar.get_height(); ax.text(bar.get_x() + bar.get_width()/2.0, yval, f'{yval:,}', ha='center', va='bottom', fontsize=14, fontweight='bold')
    plt.tight_layout(); return fig
print("  - All analytics and plots are ready.")

In [ ]:
print("\n--- 6. Building and launching the Gradio dashboard ---")
full_data_dict = {}
def get_animated_loader_html(message="Loading..."):
    return f"""<div style="display:flex; flex-direction:column; justify-content:center; align-items:center; height:750px; background-color:#f0f0f0; border-radius:10px;"><svg width="80" height="80" viewBox="0 0 38 38" xmlns="http://www.w3.org/2000/svg" stroke="#007bff"><g fill="none" fill-rule="evenodd"><g transform="translate(1 1)" stroke-width="2"><circle stroke-opacity=".5" cx="18" cy="18" r="18"/><path d="M36 18c0-9.94-8.06-18-18-18"><animateTransform attributeName="transform" type="rotate" from="0 18 18" to="360 18 18" dur="1s" repeatCount="indefinite"/></path></g></g></svg><h3 style='color:#333; margin-top:20px; font-family:sans-serif;'>{message}</h3></div>"""
def auto_initialize():
    yield {status_box: gr.update(value="Initializing..."), map_display: gr.update(value=get_animated_loader_html("Initializing Dashboard..."))}
    global full_data_dict
    if not public_url:
        yield {status_box: gr.update(value="❌ Error: Map server URL not found.")}; return
    try:
        with open(NAMES_FILE_PATH, 'r') as f: names = json.load(f)
        with open(FULL_DATA_PATH, 'r') as f: full_data_list = json.load(f)
        full_data_dict = {item['name']: item for item in full_data_list}; message = f"✅ Ready! Loaded {len(names):,} habitations."
        initial_details = handle_selection(names[0])
        yield { dropdown: gr.update(choices=names, value=names[0], interactive=True), status_box: gr.update(value=message), password_display: gr.update(value=tunnel_password),
                map_display: gr.update(value=f'<iframe id="map-iframe" src="{public_url}" style="width:100%; height:750px; border:none;" allow="geolocation"></iframe>'),
                name_out: gr.update(value=initial_details[0]), id_out: gr.update(value=initial_details[1]), status_out: gr.update(value=initial_details[2]),
                dist_out: gr.update(value=initial_details[3]), fac_out: gr.update(value=initial_details[4]), new_dist_out: gr.update(value=initial_details[5]), reduc_out: gr.update(value=initial_details[6]), }
    except Exception as e: yield {status_box: gr.update(value=f"❌ Error during initialization: {e}")}
def handle_selection(habitation_name):
    hab = full_data_dict.get(habitation_name)
    if not hab: return "", "", "", "", "", "", ""
    status_str = "🔴 Underserved" if hab.get("level") == "Underserved" else "✅ Served"; o_dist_str = f"{hab.get('orig_dist'):,.2f} m" if pd.notna(hab.get('orig_dist')) else "N/A"
    fac_str = f"{int(hab.get('near_fac_id'))}" if pd.notna(hab.get('near_fac_id')) else "N/A"; n_dist_str = f"{hab.get('net_dist_m'):,.2f} m" if pd.notna(hab.get('net_dist_m')) else "N/A"
    reduc_str = f"✨ {hab.get('net_reduc_'):,.2f} m" if pd.notna(hab.get('net_reduc_')) and hab.get('net_reduc_') > 0 else "N/A"
    return hab.get("name"), hab.get("id"), status_str, o_dist_str, fac_str, n_dist_str, reduc_str
def zoom_on_map(hab_id):
    if not hab_id: gr.Warning("No Habitation ID provided to zoom."); return gr.update(value=f'<iframe id="map-iframe" src="{public_url}" style="width:100%; height:750px; border:none;" allow="geolocation"></iframe>')
    new_url = f"{public_url}/index.html?hab_id={hab_id}"; return gr.update(value=f'<iframe id="map-iframe" src="{new_url}" style="width:100%; height:750px; border:none;" allow="geolocation"></iframe>')

with gr.Blocks(title="🏥 Healthcare Accessibility Dashboard", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🏥 Maharashtra Healthcare Accessibility Dashboard")
    with gr.Tabs():
        with gr.TabItem("Interactive Dashboard"):
            with gr.Row():
                with gr.Column(scale=1):
                    status_box = gr.Textbox(label="System Status", interactive=False); password_display = gr.Textbox(label="🔑 Map Password (Your IP Address)", interactive=False)
                    dropdown = gr.Dropdown(label="1. Select a Habitation", interactive=False)
                    with gr.Group():
                        gr.Markdown("### 📋 Selected Habitation Details"); name_out = gr.Textbox(label="Habitation Name", interactive=False); id_out = gr.Textbox(label="Habitation ID", interactive=False)
                        status_out = gr.Textbox(label="Service Status", interactive=False); dist_out = gr.Textbox(label="Original Distance", interactive=False); fac_out = gr.Textbox(label="Nearest Facility ID", interactive=False)
                        new_dist_out = gr.Textbox(label="New Proposed Distance", interactive=False); reduc_out = gr.Textbox(label="Distance Reduction", interactive=False)
                    zoom_btn = gr.Button("📍 Zoom to Habitation on Map", variant="secondary")
                with gr.Column(scale=2): map_display = gr.HTML(label="Interactive Map", value=get_animated_loader_html("Waiting for initialization..."))

        with gr.TabItem("Analysis Summary"):
            gr.Markdown("## Project Analysis & Impact Assessment")

            with gr.Accordion("Stage 1: Understanding the Current State of Accessibility", open=True):
                gr.Markdown("#### Current State: Key Metrics")
                with gr.Row():
                    gr.Textbox(label="Total Habitations Analyzed", value=f"{total_habs:,}", interactive=False); gr.Textbox(label="✅ Served Habitations", value=f"{num_served:,}", interactive=False); gr.Textbox(label="🔴 Underserved Habitations", value=f"{num_underserved:,}", interactive=False)
                gr.HTML("<hr>")
                gr.Markdown("#### Service Level Distribution\n**What it shows:** The pie chart visually represents the proportion of habitations classified as 'Served' versus 'Underserved'.\n\n**Key Insight:** A significant majority, over two-thirds of all habitations, are currently underserved, highlighting the scale of the accessibility challenge.")
                gr.Plot(value=create_service_level_plot())
                gr.HTML("<hr>")
                gr.Markdown("#### Distance Profile of the Underserved\n**What it shows:** A histogram of how far underserved habitations are from their nearest facility. The red dashed line marks the average distance for this group.\n\n**Key Insight:** The problem is most acute for a large number of habitations within the 5-20km range. While the average is high (18.7 km), the distribution has a long tail, indicating some habitations are extremely remote.")
                gr.Plot(value=create_distance_histogram())

            with gr.Accordion("Stage 2: Pathfinding Methodology for Existing Facilities", open=False):
                gr.Markdown("#### Pathfinding: Key Metrics")
                with gr.Row():
                    gr.Textbox(label="Total Paths Calculated", value=f"{paths_found:,}", interactive=False)
                    gr.Textbox(label="✅ Optimal Paths Found", value=f"{paths_optimal:,}", info="Habitation and facility are on the same, connected road network.", interactive=False)
                    gr.Textbox(label="⚠️ Fallback Paths Used", value=f"{paths_fallback:,}", info="Network was fragmented; a fallback or 'bridging' method was needed.", interactive=False)
                gr.HTML("<hr>")
                gr.Markdown("#### Breakdown of Pathfinding Methods\n**What it shows:** This chart displays the different techniques the algorithm used to find the shortest path for each habitation. This is crucial for understanding the quality and complexity of the road network data.\n\n**Key Insight:** The high number of 'Fallback' methods indicates a fragmented road network with many disconnected components. The analysis successfully navigated this by building 'bridges' and finding paths, but it highlights potential infrastructure gaps.")
                gr.Plot(value=create_path_type_plot())
            with gr.Accordion("Stage 3: Methodology for Proposing New Facilities", open=False):
                gr.Markdown("A sophisticated, multi-step **Hierarchical Clustering** approach was used to strategically place new facilities.")
                with gr.Tabs():
                    with gr.TabItem("Part A: Macro-Clustering (DBSCAN)"):
                        gr.Markdown("**What it shows:** First, we use **DBSCAN** to identify large, contiguous 'regions of need'.\n\n**Key Insight:** This initial step successfully identified **146** significant regions of need from the 58,067 underserved habitations, while filtering out 2,643 isolated habitations.")
                        with gr.Row(): gr.Textbox(label="Underserved Habitations Analyzed", value="58,067", interactive=False); gr.Textbox(label="Macro-Clusters Identified", value=f"{num_dbscan_clusters}", interactive=False); gr.Textbox(label="Isolated Habitations (Outliers)", value=f"{num_dbscan_outliers}", interactive=False)
                    with gr.TabItem("Part B: Sub-Clustering (K-Means with Elbow Method)"):
                        gr.Markdown("**What it shows:** For each large region, we use the **Elbow Method** to find the optimal number of *sub-clusters* (k).\n\n**Key Insight:** This data-driven step decides if a region needs one, two, or more new facilities. The 'elbow' in the conceptual plot below represents the point of diminishing returns, ensuring we don't propose more facilities than necessary.")
                        gr.Plot(value=create_elbow_method_explanation_plot())
                    with gr.TabItem("Part C: Core Algorithm (Dijkstra's)"):
                        gr.Markdown("**What it shows:** At the heart of facility placement is finding the shortest path on the road network. **Dijkstra's algorithm**, shown conceptually below, finds the path with the lowest total 'cost' (distance).\n\n**Key Insight:** This ensures facility locations are optimized for real-world travel, not just straight-line distance. This process is used to find the best central point ('medoid') for new facilities, resulting in **381** strategic proposals.")
                        gr.Plot(value=create_dijkstra_explanation_plot())
            with gr.Accordion("Stage 4: Quantifying the Impact of Proposed Facilities", open=True):
                gr.Markdown("#### Key Proposal Outcomes\n**What it shows:** A summary of the two most important success metrics.\n\n**Key Insight:** The proposals successfully provide a shorter travel path for over 18,000 habitations. Crucially, more than a third of these are improved so significantly that they are reclassified from 'Underserved' to 'Served'.")

                gr.Plot(value=create_impact_summary_plot())
                gr.HTML("<hr>")
                gr.Markdown("#### Shift in Distance Distribution\n**What it shows:** A 'before and after' comparison of travel distances for the 18,571 habitations that benefit from a new facility.\n\n**Key Insight:** The impact is visually clear. The green 'After' curve is dramatically shifted to the left, showing that the vast majority of these habitations are now much closer to a healthcare facility. The peak of the distribution moves from ~15km to well under 5km.")
                gr.Plot(value=create_before_after_dist_plot())
                gr.Textbox(label="Average Travel Distance Reduction for Benefiting Habitations", value=f"✨ {avg_dist_reduction_m:,.1f} meters (~{avg_dist_reduction_m/1000:.1f} km)", interactive=False)


    demo.load(fn=auto_initialize, outputs=[dropdown, status_box, password_display, map_display, name_out, id_out, status_out, dist_out, fac_out, new_dist_out, reduc_out])
    dropdown.change(fn=handle_selection, inputs=dropdown, outputs=[name_out, id_out, status_out, dist_out, fac_out, new_dist_out, reduc_out])
    zoom_btn.click(fn=zoom_on_map, inputs=[id_out], outputs=[map_display])

demo.queue().launch(share=True, debug=True)